# Flowchart DCQG Colab Notebook
Colab-safe notebook generated from the current workspace.

In [1]:
!pip install transformers sentencepiece requests bert-score


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.4 MB/s eta 0:00:00


In [2]:
!pip -q install bert-score


In [3]:
import importlib.util
print(importlib.util.find_spec("bert_score") is not None)


True


In [4]:
from bert_score import score
import torch

P, R, F1 = score(
    ["hello world"],
    ["hello world"],
    model_type="roberta-large",
    lang="en",
    device="cuda" if torch.cuda.is_available() else "cpu",
    rescale_with_baseline=True
)
print("bert_score test ok:", float(F1.mean()))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


bert_score test ok: 1.0


In [5]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/BART_commonsense_qg_output'
!mkdir -p $OUTPUT_DIR
print('OUTPUT_DIR:', OUTPUT_DIR)


Mounted at /content/drive
OUTPUT_DIR: /content/drive/MyDrive/BART_commonsense_qg_output


In [6]:
DATA_DIR = "/content/drive/MyDrive/commonsense_qg_data"

In [7]:
import os
os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)
print('Folders created: /content/data and /content/output')


Folders created: /content/data and /content/output


In [8]:
# from google.colab import files
# print('Upload these 4 files: Cosmostrain.csv, MCScript_train.xml, KQAPro_train.json, grailqa_v1.0_train.json')
# uploaded = files.upload()


In [9]:
import os, shutil

# Ensure the data directory exists
os.makedirs('/content/data', exist_ok=True)

# Directly write the uploaded files to the target directory
# The 'uploaded' variable is expected to be populated from a previous files.upload() call.
if 'uploaded' in globals():
    for filename, content in uploaded.items():
        destination_path = os.path.join('/content', 'data', filename)
        with open(destination_path, 'wb') as f:
            f.write(content)
        print(f"Wrote {filename} to {destination_path}")
else:
    print("Warning: 'uploaded' variable not found. Please ensure files.upload() was executed.")

print(os.listdir('/content/data'))


[]


## `skeletons.py`

In [10]:
%%writefile /content/skeletons.py
from __future__ import annotations

import re
from collections import Counter
from dataclasses import dataclass
from typing import Iterable, List, Optional, Sequence


RELATION_GROUPS = {
    "because": "causes",
    "why": "causes",
    "result": "causes",
    "consequence": "causes",
    "lead to": "causes",
    "caused": "causes",
    "after": "hassubevent",
    "before": "hasprerequisite",
    "when": "hassubevent",
    "during": "hassubevent",
    "first": "hasprerequisite",
    "next": "hassubevent",
    "where": "atlocation",
    "location": "atlocation",
    "place": "atlocation",
    "purpose": "usedfor",
    "for": "usedfor",
    "how": "capableof",
    "can": "capableof",
    "used": "usedfor",
    "kind": "isa",
    "type": "isa",
    "part": "partof",
    "made of": "partof",
}


def canon_relation(rel: str) -> str:
    rel = (rel or "").strip().lower().lstrip(":")
    return rel or "relatedto"


@dataclass
class EdgeSpec:
    src: str
    relation: str
    tgt: str


@dataclass
class ReasoningTrace:
    entities: List[str]
    relations: List[str]
    edges: List[tuple]
    hop_count: int
    source: str = "heuristic"


@dataclass
class Skeleton:
    id: str
    reasoning_type: str
    structure: List[EdgeSpec]
    size: int
    support: int


class TraceExtractor:
    def _extract_entities(self, text: str) -> List[str]:
        return list(
            dict.fromkeys(
                re.findall(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b", text or "")
            )
        )

    def _ordered_relations(self, text: str) -> List[str]:
        lowered = (text or "").lower()
        relations = []
        for cue, rel in sorted(RELATION_GROUPS.items(), key=lambda item: -len(item[0])):
            if cue in lowered:
                relations.append(rel)
        if relations:
            return relations[:4]
        if "?" in lowered and lowered.startswith("where"):
            return ["atlocation"]
        if "?" in lowered and lowered.startswith("why"):
            return ["causes"]
        if "?" in lowered and lowered.startswith("how"):
            return ["capableof"]
        return ["relatedto"]

    def extract_from_question(self, question: str) -> ReasoningTrace:
        relations = self._ordered_relations(question)
        entities = self._extract_entities(question)
        edges = [(f"x{idx}", rel, f"x{idx + 1}") for idx, rel in enumerate(relations)]
        return ReasoningTrace(
            entities=entities,
            relations=relations,
            edges=edges,
            hop_count=len(relations),
            source="question",
        )

    def extract_from_text(self, context: str, question: str, answer: str = "") -> ReasoningTrace:
        base = self.extract_from_question(question)
        context_entities = self._extract_entities(context)
        answer_entity = [answer] if answer else []
        entities = list(dict.fromkeys(context_entities[:4] + base.entities + answer_entity))
        if len(base.relations) == 1 and base.relations[0] == "relatedto":
            lowered = (context or "").lower()
            enriched = []
            for cue, rel in sorted(RELATION_GROUPS.items(), key=lambda item: -len(item[0])):
                if cue in lowered:
                    enriched.append(rel)
            if enriched:
                base = ReasoningTrace(
                    entities=entities,
                    relations=enriched[:4],
                    edges=[(f"x{idx}", rel, f"x{idx + 1}") for idx, rel in enumerate(enriched[:4])],
                    hop_count=len(enriched[:4]),
                    source="context+question",
                )
        else:
            base.entities = entities
        return base

    def extract_from_path(self, path, graph) -> ReasoningTrace:
        relations = [canon_relation(edge.rel) for edge in getattr(path, "edges", [])]
        entities = [graph.nodes[node_id].label for node_id in getattr(path, "nodes", []) if node_id in graph.nodes]
        edges = []
        for edge in getattr(path, "edges", []):
            src = graph.nodes[edge.src].label if edge.src in graph.nodes else edge.src
            tgt = graph.nodes[edge.tgt].label if edge.tgt in graph.nodes else edge.tgt
            edges.append((src, canon_relation(edge.rel), tgt))
        return ReasoningTrace(
            entities=entities,
            relations=relations or ["relatedto"],
            edges=edges,
            hop_count=len(relations or ["relatedto"]),
            source="path",
        )


class SkeletonMiner:
    def learn_from_traces(self, traces: Iterable[ReasoningTrace], min_support: int = 2, max_hops: int = 4) -> List[Skeleton]:
        patterns = Counter()
        for trace in traces:
            rels = tuple(canon_relation(rel) for rel in trace.relations[:max_hops])
            if rels:
                patterns[rels] += 1
        skeletons: List[Skeleton] = []
        for idx, (pattern, support) in enumerate(sorted(patterns.items(), key=lambda item: (-item[1], item[0]))):
            if support < min_support:
                continue
            structure = [EdgeSpec("X", rel, "Y") for rel in pattern]
            skeletons.append(Skeleton(id=f"sk_{idx}", reasoning_type=self._infer_type(pattern), structure=structure, size=len(structure), support=support))
        return skeletons

    def learn_from_questions(self, questions: Sequence[str], min_support: int = 2, max_hops: int = 4) -> List[Skeleton]:
        extractor = TraceExtractor()
        traces = [extractor.extract_from_question(question) for question in questions]
        return self.learn_from_traces(traces, min_support=min_support, max_hops=max_hops)

    def _infer_type(self, pattern: Sequence[str]) -> str:
        if any(rel in {"causes"} for rel in pattern):
            return "causal"
        if any(rel in {"hassubevent", "hasprerequisite"} for rel in pattern):
            return "temporal"
        if any(rel in {"usedfor", "capableof"} for rel in pattern):
            return "purpose"
        if any(rel == "atlocation" for rel in pattern):
            return "location"
        if any(rel == "isa" for rel in pattern):
            return "taxonomy"
        return "other"


class DifficultyController:
    def __init__(self, skeleton_bank: Sequence[Skeleton]) -> None:
        self.skeleton_bank = list(skeleton_bank)

    def candidates(self, difficulty_level: int) -> List[Skeleton]:
        exact = [skeleton for skeleton in self.skeleton_bank if skeleton.size == difficulty_level]
        if exact:
            return exact
        close = [skeleton for skeleton in self.skeleton_bank if abs(skeleton.size - difficulty_level) <= 1]
        return close or list(self.skeleton_bank)

    def select(
        self,
        difficulty_level: int,
        question: str = "",
        preferred_relations: Optional[Sequence[str]] = None,
        preferred_type: str = "",
    ) -> Optional[Skeleton]:
        candidates = self.candidates(difficulty_level)
        if not candidates:
            return None
        if not question and not preferred_relations and not preferred_type:
            return sorted(candidates, key=lambda item: (-item.support, item.id))[0]
        trace = TraceExtractor().extract_from_question(question) if question else None
        trace_sequence = list(preferred_relations or (trace.relations if trace is not None else []))
        trace_relations = set(trace_sequence)
        scored = []
        for skeleton in candidates:
            sk_rels = [spec.relation for spec in skeleton.structure]
            overlap = len(set(sk_rels) & trace_relations)
            prefix = 0
            if trace_sequence:
                prefix = sum(1 for left, right in zip(sk_rels, trace_sequence) if left == right)
            type_bonus = 1 if preferred_type and skeleton.reasoning_type == preferred_type else 0
            scored.append(
                (type_bonus, overlap, prefix, skeleton.support, -abs(skeleton.size - difficulty_level), skeleton)
            )
        scored.sort(key=lambda item: (item[0], item[1], item[2]), reverse=True)
        return scored[0][-1]


def skeleton_to_id(skeleton: Optional[Skeleton], max_buckets: int = 64) -> int:
    if skeleton is None:
        return 0
    try:
        return int(skeleton.id.split("_")[-1]) % max_buckets
    except Exception:
        return 0


def form_to_id(form: Optional[List[int]], max_buckets: int = 64) -> int:
    if not form:
        return 0
    return int(sum(form) / max(1, len(form))) % max_buckets


Writing /content/skeletons.py


## `hsmm.py`

In [11]:
%%writefile /content/hsmm.py
from __future__ import annotations

import random
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


def tokenize(text: str) -> List[str]:
    return [tok for tok in text.lower().split() if tok]


class HSMM(nn.Module):
    def __init__(self, vocab_size: int, n_states: int = 16, max_duration: int = 6, embed_dim: int = 96) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.n_states = n_states
        self.max_duration = max_duration
        self.state_embedding = nn.Embedding(n_states, embed_dim)
        self.transition = nn.Parameter(torch.randn(n_states, n_states) * 0.02)
        self.start = nn.Parameter(torch.zeros(n_states))
        self.emission = nn.Linear(embed_dim, vocab_size)
        self.duration = nn.Linear(embed_dim, max_duration)

    def emission_logprob(self, state_ids: torch.Tensor) -> torch.Tensor:
        emb = self.state_embedding(state_ids)
        return F.log_softmax(self.emission(emb), dim=-1)

    def duration_logprob(self, state_ids: torch.Tensor) -> torch.Tensor:
        emb = self.state_embedding(state_ids)
        return F.log_softmax(self.duration(emb), dim=-1)


def semi_markov_loglikelihood(model: HSMM, seq: torch.Tensor) -> torch.Tensor:
    device = seq.device
    steps = int(seq.numel())
    states = int(model.n_states)
    max_duration = int(model.max_duration)
    if steps == 0:
        return torch.tensor(0.0, device=device)
    state_ids = torch.arange(states, device=device, dtype=torch.long)
    emission = model.emission_logprob(state_ids)
    duration = model.duration_logprob(state_ids)
    transition = F.log_softmax(model.transition, dim=-1)
    start = F.log_softmax(model.start, dim=-1)
    token_lp = emission[:, seq.long()].transpose(0, 1).contiguous()
    csum = torch.cat([torch.zeros(1, states, device=device), torch.cumsum(token_lp, dim=0)], dim=0)
    alpha = torch.full((steps + 1, states), -1e9, device=device)
    for end in range(1, steps + 1):
        terms = []
        for duration_len in range(1, min(max_duration, end) + 1):
            start_idx = end - duration_len
            seg_emit = csum[end] - csum[start_idx]
            seg_dur = duration[:, duration_len - 1]
            prev = start if start_idx == 0 else torch.logsumexp(alpha[start_idx].unsqueeze(1) + transition, dim=0)
            terms.append(prev + seg_dur + seg_emit)
        alpha[end] = torch.logsumexp(torch.stack(terms, dim=0), dim=0)
    return torch.logsumexp(alpha[steps], dim=0)


def viterbi_segment(model: HSMM, token_tensor: torch.Tensor) -> List[Tuple[int, int, int]]:
    device = token_tensor.device
    tokens = token_tensor[0].long()
    steps = int(tokens.numel())
    if steps == 0:
        return []
    state_ids = torch.arange(model.n_states, device=device, dtype=torch.long)
    emission = model.emission_logprob(state_ids)
    emit_scores = emission[:, tokens].transpose(0, 1).contiguous()
    dp = torch.full((steps, model.n_states), -1e9, device=device)
    back = torch.zeros((steps, model.n_states), dtype=torch.long, device=device)
    dp[0] = emit_scores[0] + F.log_softmax(model.start, dim=-1)
    transition = F.log_softmax(model.transition, dim=-1)
    for step in range(1, steps):
        for state in range(model.n_states):
            scores = dp[step - 1] + transition[:, state]
            prev = torch.argmax(scores)
            dp[step, state] = scores[prev] + emit_scores[step, state]
            back[step, state] = prev
    states = []
    state = int(torch.argmax(dp[-1]).item())
    for step in reversed(range(steps)):
        states.append(state)
        state = int(back[step, state].item())
    states.reverse()
    segments = []
    cur_state = states[0]
    start = 0
    for idx in range(1, len(states)):
        if states[idx] != cur_state:
            segments.append((cur_state, start, idx))
            cur_state = states[idx]
            start = idx
    segments.append((cur_state, start, len(states)))
    return segments


@dataclass
class HSMMArtifacts:
    model: HSMM
    vocab: Dict[str, int]
    pool: "ExpressiveFormPool"


class ExpressiveFormPool:
    def __init__(self) -> None:
        self.forms: List[List[int]] = []
        self.by_length: Dict[int, List[List[int]]] = {}
        self.by_signature: Dict[str, List[List[int]]] = {}
        self.by_signature_length: Dict[Tuple[str, int], List[List[int]]] = {}

    def add(self, segments: Sequence[Tuple[int, int, int]], signature: str = "") -> None:
        if not segments:
            return
        form = [state for state, _, _ in segments]
        self.forms.append(form)
        self.by_length.setdefault(len(form), []).append(form)
        signature = signature or "global"
        self.by_signature.setdefault(signature, []).append(form)
        self.by_signature_length.setdefault((signature, len(form)), []).append(form)

    def sample(self, preferred_length: Optional[int] = None, signature: str = "") -> List[int]:
        signature = signature or "global"
        if preferred_length is not None and (signature, preferred_length) in self.by_signature_length:
            candidates = self.by_signature_length[(signature, preferred_length)]
            if candidates:
                return random.choice(candidates)
        if signature in self.by_signature and self.by_signature[signature]:
            return random.choice(self.by_signature[signature])
        if preferred_length is not None and preferred_length in self.by_length and self.by_length[preferred_length]:
            return random.choice(self.by_length[preferred_length])
        if not self.forms:
            return [0]
        return random.choice(self.forms)


class HSMMTrainer:
    def __init__(self, n_states: int = 16, max_duration: int = 6, embed_dim: int = 96, device: str = "cpu") -> None:
        self.n_states = n_states
        self.max_duration = max_duration
        self.embed_dim = embed_dim
        self.device = device

    def fit(
        self,
        questions: Sequence[str],
        epochs: int = 3,
        lr: float = 1e-3,
        signatures: Optional[Sequence[str]] = None,
    ) -> HSMMArtifacts:
        vocab = self._build_vocab(questions)
        if not vocab:
            vocab = {"<unk>": 0}
        dataset = []
        dataset_signatures = []
        raw_signatures = list(signatures or [])
        for idx, question in enumerate(questions):
            if not tokenize(question):
                continue
            dataset.append(self._tensorize(question, vocab))
            dataset_signatures.append(raw_signatures[idx] if idx < len(raw_signatures) else "global")
        if not dataset:
            dataset = [torch.tensor([0], dtype=torch.long)]
            dataset_signatures = ["global"]
        model = HSMM(vocab_size=len(vocab), n_states=self.n_states, max_duration=self.max_duration, embed_dim=self.embed_dim).to(self.device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        model.train()
        for _ in range(epochs):
            for seq in dataset:
                seq = seq.to(self.device)
                loss = -semi_markov_loglikelihood(model, seq)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        pool = ExpressiveFormPool()
        model.eval()
        with torch.no_grad():
            for idx, seq in enumerate(dataset):
                signature = dataset_signatures[idx] if idx < len(dataset_signatures) else "global"
                pool.add(viterbi_segment(model, seq.unsqueeze(0).to(self.device)), signature=signature)
        return HSMMArtifacts(model=model, vocab=vocab, pool=pool)

    def segment(self, question: str, artifacts: HSMMArtifacts) -> List[Tuple[int, int, int]]:
        token_tensor = self._tensorize(question, artifacts.vocab).unsqueeze(0).to(self.device)
        with torch.no_grad():
            return viterbi_segment(artifacts.model, token_tensor)

    def _build_vocab(self, questions: Sequence[str]) -> Dict[str, int]:
        counter = Counter()
        for question in questions:
            counter.update(tokenize(question))
        vocab = {"<unk>": 0}
        for token, _ in counter.most_common():
            vocab[token] = len(vocab)
        return vocab

    def _tensorize(self, question: str, vocab: Dict[str, int]) -> torch.Tensor:
        ids = [vocab.get(token, 0) for token in tokenize(question)]
        return torch.tensor(ids or [0], dtype=torch.long)


Writing /content/hsmm.py


## `graphs_build.py`

In [12]:
%%writefile /content/graphs_build.py
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import string
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import requests
import torch
import torch.nn as nn


STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "has", "he",
    "in", "is", "it", "its", "of", "on", "that", "the", "to", "was", "were", "will",
    "with", "she", "they", "their", "his", "her", "this", "those", "these", "or",
}

RELATION_CANON = {
    "AtLocation": "atlocation",
    "LocatedNear": "atlocation",
    "Causes": "causes",
    "CausesDesire": "causes",
    "MotivatedByGoal": "causes",
    "IsA": "isa",
    "InstanceOf": "isa",
    "PartOf": "partof",
    "UsedFor": "usedfor",
    "CapableOf": "capableof",
    "RelatedTo": "relatedto",
    "SimilarTo": "relatedto",
    "Synonym": "relatedto",
    "HasSubevent": "hassubevent",
    "HasPrerequisite": "hasprerequisite",
    "xNeed": "causes",
    "xIntent": "usedfor",
    "xEffect": "causes",
    "oEffect": "causes",
    "xWant": "usedfor",
    "oWant": "usedfor",
}

IMPORTANT_RELATIONS = {
    "atlocation",
    "capableof",
    "causes",
    "hassubevent",
    "hasprerequisite",
    "isa",
    "partof",
    "relatedto",
    "usedfor",
}

RELATION_WEIGHTS = defaultdict(
    lambda: 1.0,
    {
        "causes": 1.8,
        "hassubevent": 1.6,
        "hasprerequisite": 1.6,
        "usedfor": 1.5,
        "capableof": 1.5,
        "isa": 1.4,
        "partof": 1.3,
        "atlocation": 1.2,
        "relatedto": 0.8,
    },
)


def normalize_text(text: str) -> str:
    text = (text or "").strip().lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()


def normalize_concept(text: str) -> str:
    return normalize_text(text).replace(" ", "_")


def canonical_relation(rel: str) -> str:
    key = (rel or "").strip().lstrip(":")
    if key in RELATION_CANON:
        return RELATION_CANON[key]
    if key.lower() in RELATION_CANON:
        return RELATION_CANON[key.lower()]
    return key.lower() or "relatedto"


def simple_tokenize(text: str) -> List[str]:
    return [tok for tok in re.findall(r"[A-Za-z][A-Za-z\-']*", text.lower()) if tok]


def lexical_overlap(a: str, b: str) -> float:
    a_tokens = set(simple_tokenize(a))
    b_tokens = set(simple_tokenize(b))
    if not a_tokens or not b_tokens:
        return 0.0
    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


@dataclass
class Node:
    id: str
    label: str
    node_type: str = "context"
    meta: Dict[str, str] = field(default_factory=dict)


@dataclass
class Edge:
    src: str
    rel: str
    tgt: str
    weight: float = 1.0
    meta: Dict[str, str] = field(default_factory=dict)


@dataclass
class AMRGraph:
    concepts: List[str]
    relations: List[Tuple[str, str, str]]


@dataclass
class InferentialGraph:
    nodes: Dict[str, Node] = field(default_factory=dict)
    edges: List[Edge] = field(default_factory=list)

    def add_node(self, node: Node) -> None:
        self.nodes[node.id] = node

    def add_edge(self, edge: Edge) -> None:
        self.edges.append(edge)

    def build_index(self) -> Dict[str, str]:
        return {normalize_text(node.label): nid for nid, node in self.nodes.items()}


@dataclass
class ReasoningPath:
    nodes: List[str]
    edges: List[Edge]
    score: float

    @property
    def hop_count(self) -> int:
        return len(self.edges)


@dataclass
class RetrievedSubgraph:
    graph: InferentialGraph
    paths: List[ReasoningPath]
    path_nodes: List[str]
    hop_count: int


@dataclass
class MCSMatchState:
    mapping: List[Tuple[int, str]]
    used_nodes: set
    score: float
    frontier: List[str]


class AMRProxyParser:
    def __init__(self) -> None:
        self._parser = None
        try:
            import amrlib  # type: ignore

            model_dir = os.getenv("DCQG_AMR_MODEL_DIR", "").strip()
            self._parser = amrlib.load_stog_model(model_dir=model_dir) if model_dir else amrlib.load_stog_model()
        except Exception:
            self._parser = None

    def parse(self, text: str) -> AMRGraph:
        if self._parser is not None:
            parsed = self._parse_with_amrlib(text)
            if parsed is not None:
                return parsed
        return self._parse_proxy(text)

    def _parse_with_amrlib(self, text: str) -> Optional[AMRGraph]:
        try:
            import penman  # type: ignore

            graphs = self._parser.parse_sents([text])
            if not graphs:
                return None
            g = penman.decode(graphs[0])
            var_to_concept = {inst.source: inst.target for inst in g.instances()}
            concepts = list(dict.fromkeys(var_to_concept.values()))
            relations = []
            for edge in g.edges():
                src = var_to_concept.get(edge.source, edge.source)
                tgt = var_to_concept.get(edge.target, edge.target)
                relations.append((src, edge.role.lstrip(":"), tgt))
            if concepts:
                return AMRGraph(concepts=concepts, relations=relations)
        except Exception:
            return None
        return None

    def _parse_proxy(self, text: str) -> AMRGraph:
        sentences = re.split(r"[.!?]+", text)
        concepts: List[str] = []
        relations: List[Tuple[str, str, str]] = []
        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue
            entity_mentions = re.findall(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b", sentence)
            content_words = [
                tok for tok in simple_tokenize(sentence)
                if tok not in STOPWORDS and len(tok) > 2
            ]
            local_concepts = entity_mentions[:]
            for tok in content_words:
                if tok not in local_concepts:
                    local_concepts.append(tok)
                if len(local_concepts) >= 8:
                    break
            for concept in local_concepts:
                if concept not in concepts:
                    concepts.append(concept)
            rel = self._infer_relation(sentence)
            for src, tgt in zip(local_concepts, local_concepts[1:]):
                relations.append((src, rel, tgt))
        if not concepts:
            concepts = ["context"]
        if not relations and len(concepts) > 1:
            relations = [(concepts[i], "relatedto", concepts[i + 1]) for i in range(len(concepts) - 1)]
        return AMRGraph(concepts=concepts, relations=relations)

    def _infer_relation(self, sentence: str) -> str:
        s = sentence.lower()
        if any(tok in s for tok in ["because", "therefore", "so", "hence"]):
            return "causes"
        if any(tok in s for tok in ["before", "after", "then", "next"]):
            return "hassubevent"
        if any(tok in s for tok in ["in order", " to ", " for "]):
            return "usedfor"
        if any(tok in s for tok in [" in ", " at ", "inside", "near"]):
            return "atlocation"
        return "relatedto"


def build_context_graph(amr: AMRGraph) -> InferentialGraph:
    graph = InferentialGraph()
    label_to_id: Dict[str, str] = {}
    for idx, concept in enumerate(amr.concepts):
        node_id = f"c{idx}"
        label_to_id[concept] = node_id
        graph.add_node(Node(id=node_id, label=concept, node_type="context"))
    for src_label, rel, tgt_label in amr.relations:
        if src_label not in label_to_id or tgt_label not in label_to_id:
            continue
        canon = canonical_relation(rel)
        graph.add_edge(Edge(src=label_to_id[src_label], rel=canon, tgt=label_to_id[tgt_label], weight=RELATION_WEIGHTS[canon]))
    return graph


def extract_entities(text: str, answer: str = "") -> List[str]:
    entities = re.findall(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b", text)
    answer_tokens = [tok for tok in simple_tokenize(answer) if tok not in STOPWORDS]
    tokens = [tok for tok in simple_tokenize(text) if tok not in STOPWORDS and len(tok) > 3]
    salient = []
    for tok in tokens:
        if any(tok.startswith(ans[:4]) for ans in answer_tokens if len(ans) >= 4):
            salient.append(tok)
    merged = list(dict.fromkeys(entities + salient[:4] + tokens[:8] + ([answer] if answer else [])))
    return [item for item in merged if item]


class ConceptNetBackend:
    BASE_URL = "https://api.conceptnet.io/query"

    def __init__(self, cache_dir: str = "data/cache/kb", timeout: int = 5) -> None:
        self.cache_dir = cache_dir
        self.timeout = timeout
        os.makedirs(self.cache_dir, exist_ok=True)

    def retrieve(self, concepts: Sequence[str], limit: int = 50, hops: int = 2) -> List[Tuple[str, str, str]]:
        triples: List[Tuple[str, str, str]] = []
        seen = set()
        frontier = list(dict.fromkeys(normalize_concept(c) for c in concepts if c))
        for _ in range(max(1, hops)):
            next_frontier = []
            for concept in frontier:
                for triple in self._fetch(concept):
                    canon = (triple[0], canonical_relation(triple[1]), triple[2])
                    if canon in seen:
                        continue
                    seen.add(canon)
                    triples.append(canon)
                    if canon[1] in IMPORTANT_RELATIONS:
                        next_frontier.append(canon[2])
                    if len(triples) >= limit:
                        return triples
            frontier = next_frontier
            if not frontier:
                break
        return triples

    def _fetch(self, concept: str) -> List[Tuple[str, str, str]]:
        key = hashlib.md5(concept.encode("utf-8")).hexdigest()
        cache_file = os.path.join(self.cache_dir, f"cn_{key}.json")
        if os.path.exists(cache_file):
            with open(cache_file, "r", encoding="utf-8") as handle:
                return json.load(handle)
        try:
            response = requests.get(
                self.BASE_URL,
                params={"node": f"/c/en/{concept}", "limit": 20},
                timeout=self.timeout,
            )
            data = response.json()
        except Exception:
            return []
        triples = []
        for edge in data.get("edges", []):
            rel = canonical_relation(edge.get("rel", {}).get("label", "relatedto"))
            src = normalize_concept(edge.get("start", {}).get("label", ""))
            tgt = normalize_concept(edge.get("end", {}).get("label", ""))
            if src and tgt:
                triples.append((src, rel, tgt))
        with open(cache_file, "w", encoding="utf-8") as handle:
            json.dump(triples, handle)
        return triples


class AtomicBackend:
    def __init__(self, path: Optional[str] = None) -> None:
        self.triples: List[Tuple[str, str, str]] = []
        if path and os.path.exists(path):
            self.triples = self._load(path)

    def _load(self, path: str) -> List[Tuple[str, str, str]]:
        triples = []
        with open(path, "r", encoding="utf-8") as handle:
            for line in handle:
                line = line.strip()
                if not line:
                    continue
                if line.startswith("{"):
                    obj = json.loads(line)
                    head = normalize_concept(obj.get("head", ""))
                    rel = canonical_relation(obj.get("rel", "relatedto"))
                    tail = normalize_concept(obj.get("tail", ""))
                else:
                    parts = line.split("\t")
                    if len(parts) < 3:
                        continue
                    head = normalize_concept(parts[0])
                    rel = canonical_relation(parts[1])
                    tail = normalize_concept(parts[2])
                if head and tail:
                    triples.append((head, rel, tail))
        return triples

    def retrieve(self, concepts: Sequence[str], limit: int = 30) -> List[Tuple[str, str, str]]:
        concept_set = {normalize_concept(item) for item in concepts if item}
        out = []
        for head, rel, tail in self.triples:
            if head in concept_set or tail in concept_set:
                out.append((head, rel, tail))
                if len(out) >= limit:
                    break
        return out


class LocalTSVBackend:
    def __init__(self, path: Optional[str] = None) -> None:
        self.index: Dict[str, List[Tuple[str, str, str]]] = defaultdict(list)
        if path and os.path.exists(path):
            with open(path, "r", encoding="utf-8") as handle:
                for line in handle:
                    parts = line.strip().split("\t")
                    if len(parts) < 3:
                        continue
                    head = normalize_concept(parts[0])
                    rel = canonical_relation(parts[1])
                    tail = normalize_concept(parts[2])
                    self.index[head].append((head, rel, tail))
                    self.index[tail].append((head, rel, tail))

    def retrieve(self, concepts: Sequence[str], limit: int = 30) -> List[Tuple[str, str, str]]:
        triples = []
        seen = set()
        for concept in concepts:
            key = normalize_concept(concept)
            for triple in self.index.get(key, []):
                if triple in seen:
                    continue
                seen.add(triple)
                triples.append(triple)
                if len(triples) >= limit:
                    return triples
        return triples


class CompositeKBBackend:
    def __init__(
        self,
        conceptnet: Optional[ConceptNetBackend] = None,
        atomic: Optional[AtomicBackend] = None,
        local_tsv: Optional[LocalTSVBackend] = None,
    ) -> None:
        self.conceptnet = conceptnet or ConceptNetBackend()
        self.atomic = atomic or AtomicBackend(os.getenv("DCQG_ATOMIC_PATH"))
        self.local_tsv = local_tsv or LocalTSVBackend(os.getenv("DCQG_LOCAL_TSV"))

    def retrieve(self, concepts: Sequence[str], conceptnet_limit: int = 50, atomic_limit: int = 30) -> List[Tuple[str, str, str]]:
        triples = []
        seen = set()
        sources = [
            self.local_tsv.retrieve(concepts, limit=conceptnet_limit),
            self.atomic.retrieve(concepts, limit=atomic_limit),
            self.conceptnet.retrieve(concepts, limit=conceptnet_limit, hops=2),
        ]
        for group in sources:
            for triple in group:
                if triple in seen:
                    continue
                seen.add(triple)
                triples.append(triple)
        return triples


def build_inferential_graph(context_graph: InferentialGraph, triples: Sequence[Tuple[str, str, str]]) -> InferentialGraph:
    graph = InferentialGraph(nodes=dict(context_graph.nodes), edges=list(context_graph.edges))
    index = graph.build_index()

    def ensure_node(label: str, node_type: str) -> str:
        key = normalize_text(label.replace("_", " "))
        if key in index:
            return index[key]
        node_id = f"n{len(graph.nodes)}"
        graph.add_node(Node(id=node_id, label=label.replace("_", " "), node_type=node_type))
        index[key] = node_id
        return node_id

    seen_edges = {(edge.src, edge.rel, edge.tgt) for edge in graph.edges}
    for head, rel, tail in triples:
        src = ensure_node(head, "commonsense")
        tgt = ensure_node(tail, "commonsense")
        canon = canonical_relation(rel)
        key = (src, canon, tgt)
        if key in seen_edges:
            continue
        graph.add_edge(Edge(src=src, rel=canon, tgt=tgt, weight=RELATION_WEIGHTS[canon]))
        seen_edges.add(key)
    return graph


def prune_inferential_graph(graph: InferentialGraph, answer_text: str, keep_relations: Optional[Iterable[str]] = None) -> InferentialGraph:
    keep_relations = set(keep_relations or IMPORTANT_RELATIONS)
    keep_nodes = set()
    answer_text = normalize_text(answer_text)
    adjacency = defaultdict(set)
    for edge in graph.edges:
        adjacency[edge.src].add(edge.tgt)
        adjacency[edge.tgt].add(edge.src)
        if edge.rel in keep_relations:
            keep_nodes.add(edge.src)
            keep_nodes.add(edge.tgt)
    for node_id, node in graph.nodes.items():
        if answer_text and answer_text in normalize_text(node.label):
            keep_nodes.add(node_id)
            keep_nodes.update(adjacency.get(node_id, set()))
    context_nodes = [nid for nid, node in graph.nodes.items() if node.node_type == "context"]
    context_degree = Counter()
    for edge in graph.edges:
        if edge.src in graph.nodes and graph.nodes[edge.src].node_type == "context":
            context_degree[edge.src] += 1
        if edge.tgt in graph.nodes and graph.nodes[edge.tgt].node_type == "context":
            context_degree[edge.tgt] += 1
    for node_id, _ in context_degree.most_common(2):
        keep_nodes.add(node_id)
        keep_nodes.update(adjacency.get(node_id, set()))
    if not keep_nodes:
        return graph
    changed = True
    while changed:
        changed = False
        for edge in graph.edges:
            if edge.src in keep_nodes or edge.tgt in keep_nodes:
                if edge.src not in keep_nodes:
                    keep_nodes.add(edge.src)
                    changed = True
                if edge.tgt not in keep_nodes:
                    keep_nodes.add(edge.tgt)
                    changed = True
    nodes = {nid: node for nid, node in graph.nodes.items() if nid in keep_nodes}
    edges = [edge for edge in graph.edges if edge.src in nodes and edge.tgt in nodes]
    return InferentialGraph(nodes=nodes, edges=edges)


def relation_matches(expected: str, observed: str) -> bool:
    expected = canonical_relation(expected)
    observed = canonical_relation(observed)
    if expected == observed:
        return True
    alias = {
        "causes": {"causes", "hassubevent", "hasprerequisite"},
        "usedfor": {"usedfor", "capableof"},
        "atlocation": {"atlocation", "relatedto"},
        "relatedto": {"relatedto", "isa", "partof"},
    }
    return observed in alias.get(expected, {expected})


def _adjacency(graph: InferentialGraph) -> Dict[str, List[Edge]]:
    adj: Dict[str, List[Edge]] = defaultdict(list)
    for edge in graph.edges:
        adj[edge.src].append(edge)
    return adj


def _undirected_adjacency(graph: InferentialGraph) -> Dict[str, List[Tuple[str, Edge]]]:
    adj: Dict[str, List[Tuple[str, Edge]]] = defaultdict(list)
    for edge in graph.edges:
        adj[edge.src].append((edge.tgt, edge))
        adj[edge.tgt].append((edge.src, edge))
    return adj


def _find_answer_nodes(graph: InferentialGraph, answer_text: str) -> List[str]:
    answer_text = normalize_text(answer_text)
    return [
        node_id
        for node_id, node in graph.nodes.items()
        if answer_text and answer_text in normalize_text(node.label)
    ]


def _node_salience(graph: InferentialGraph, node_id: str, answer_nodes: Sequence[str]) -> float:
    node = graph.nodes[node_id]
    score = 0.0
    if node.node_type == "context":
        score += 0.2
    if node_id in answer_nodes:
        score += 1.0
    label = normalize_text(node.label)
    if any(label in normalize_text(graph.nodes[target].label) or normalize_text(graph.nodes[target].label) in label for target in answer_nodes):
        score += 0.6
    return score


def _seed_start_nodes(graph: InferentialGraph, answer_nodes: Sequence[str]) -> List[str]:
    scored = []
    for node_id, node in graph.nodes.items():
        if node_id in answer_nodes:
            continue
        score = _node_salience(graph, node_id, answer_nodes)
        if node.node_type == "context":
            score += 0.4
        scored.append((score, node_id))
    scored.sort(reverse=True)
    if not scored:
        return list(graph.nodes.keys())
    return [node_id for _, node_id in scored[: max(6, min(12, len(scored)))]]


def _skeleton_pattern(skeleton, difficulty: int) -> List[str]:
    pattern = [canonical_relation(spec.relation) for spec in getattr(skeleton, "structure", [])]
    if not pattern:
        return ["relatedto"] * max(1, difficulty)
    return pattern


def _promise_bound(state: MCSMatchState, pattern: Sequence[str]) -> float:
    remaining = max(0, len(pattern) - len(state.mapping))
    return state.score + 1.8 * remaining


def _glsearch_mcs(graph: InferentialGraph, pattern: Sequence[str], answer_nodes: Sequence[str], topk: int = 5) -> List[ReasoningPath]:
    if not pattern:
        return []
    answer_set = set(answer_nodes)
    undirected = _undirected_adjacency(graph)
    seeds = _seed_start_nodes(graph, answer_nodes)
    beam: List[MCSMatchState] = [
        MCSMatchState(mapping=[], used_nodes={seed}, score=_node_salience(graph, seed, answer_nodes), frontier=[seed])
        for seed in seeds
    ]
    completed: List[ReasoningPath] = []

    while beam:
        expanded: List[MCSMatchState] = []
        for state in beam:
            step = len(state.mapping)
            if step >= len(pattern):
                if state.frontier and state.frontier[-1] in answer_set:
                    nodes = state.frontier[:]
                    edges = [edge for _, _, edge in state.mapping]
                    completed.append(ReasoningPath(nodes=nodes, edges=edges, score=state.score))
                continue
            expected = canonical_relation(pattern[step])
            tail = state.frontier[-1]
            for neighbor, edge in undirected.get(tail, []):
                if neighbor in state.used_nodes:
                    continue
                observed = canonical_relation(edge.rel)
                if not relation_matches(expected, observed):
                    continue
                bonus = 0.25 if observed == expected else 0.1
                bonus += 0.15 * _node_salience(graph, neighbor, answer_nodes)
                if step == len(pattern) - 1 and neighbor in answer_set:
                    bonus += 0.6
                directed = edge if edge.src == tail else Edge(src=tail, rel=edge.rel, tgt=neighbor, weight=edge.weight, meta=edge.meta)
                expanded.append(
                    MCSMatchState(
                        mapping=state.mapping + [(step, neighbor, directed)],
                        used_nodes=state.used_nodes | {neighbor},
                        score=state.score + math.log(RELATION_WEIGHTS[observed] + 1e-6) + bonus,
                        frontier=state.frontier + [neighbor],
                    )
                )
        if not expanded:
            break
        expanded.sort(key=lambda item: (_promise_bound(item, pattern), item.score), reverse=True)
        beam = expanded[: max(12, topk * 4)]

    completed.sort(key=lambda item: item.score, reverse=True)
    return completed[:topk]


def retrieve_reasoning_subgraph(graph: InferentialGraph, answer_text: str, skeleton, difficulty: int, topk: int = 5) -> RetrievedSubgraph:
    target_nodes = _find_answer_nodes(graph, answer_text)
    if not target_nodes:
        return RetrievedSubgraph(graph=graph, paths=[], path_nodes=[], hop_count=0)
    pattern = _skeleton_pattern(skeleton, difficulty)
    mcs_paths = _glsearch_mcs(graph, pattern, target_nodes, topk=topk)
    if mcs_paths:
        node_labels = [graph.nodes[node_id].label for node_id in mcs_paths[0].nodes if node_id in graph.nodes]
        return RetrievedSubgraph(
            graph=graph,
            paths=mcs_paths,
            path_nodes=node_labels,
            hop_count=mcs_paths[0].hop_count,
        )
    adj = _adjacency(graph)
    start_nodes = _seed_start_nodes(graph, target_nodes)
    min_hops = max(1, min(len(pattern), difficulty))
    max_hops = max(min_hops, min(max(len(pattern) + 1, difficulty + 1), 5))
    beam = [(0.0, [nid], [], 0) for nid in start_nodes]
    finished: List[ReasoningPath] = []
    while beam:
        next_beam = []
        for score, nodes, edges, step in beam:
            last = nodes[-1]
            if step >= max_hops:
                if last in target_nodes and len(edges) >= min_hops:
                    finished.append(ReasoningPath(nodes=nodes, edges=edges, score=score))
                continue
            if last in target_nodes and len(edges) >= min_hops:
                finished.append(ReasoningPath(nodes=nodes, edges=edges, score=score + 0.5))
                continue
            expected = pattern[min(step, len(pattern) - 1)]
            for edge in adj.get(last, []):
                if edge.tgt in nodes:
                    continue
                relaxed_expected = pattern[min(len(pattern) - 1, max(0, step - 1))]
                if not relation_matches(expected, edge.rel) and not relation_matches(relaxed_expected, edge.rel):
                    continue
                rel = canonical_relation(edge.rel)
                bonus = 0.2 if rel == expected else 0.0
                bonus += 0.1 * _node_salience(graph, edge.tgt, target_nodes)
                if edge.tgt in target_nodes and step + 1 >= min_hops:
                    bonus += 0.4
                next_beam.append((score + math.log(RELATION_WEIGHTS[rel] + 1e-6) + bonus, nodes + [edge.tgt], edges + [edge], step + 1))
        if not next_beam:
            break
        next_beam.sort(key=lambda item: item[0], reverse=True)
        beam = next_beam[: max(8, topk * 2)]
    if not finished:
        finished = _fallback_beam_search(graph, start_nodes, target_nodes, max_hops=max_hops, topk=topk)
    finished.sort(key=lambda item: item.score, reverse=True)
    finished = finished[:topk]
    node_labels = [graph.nodes[node_id].label for node_id in finished[0].nodes if node_id in graph.nodes] if finished else []
    return RetrievedSubgraph(graph=graph, paths=finished, path_nodes=node_labels, hop_count=finished[0].hop_count if finished else 0)


def _fallback_beam_search(graph: InferentialGraph, start_nodes: Sequence[str], target_nodes: Sequence[str], max_hops: int, topk: int) -> List[ReasoningPath]:
    targets = set(target_nodes)
    adj = _adjacency(graph)
    frontier = [(0.0, [nid], []) for nid in start_nodes]
    results: List[ReasoningPath] = []
    for _ in range(max_hops):
        new_frontier = []
        for score, nodes, edges in frontier:
            for edge in adj.get(nodes[-1], []):
                if edge.tgt in nodes:
                    continue
                rel = canonical_relation(edge.rel)
                new_score = score + math.log(RELATION_WEIGHTS[rel] + 1e-6)
                new_nodes = nodes + [edge.tgt]
                new_edges = edges + [edge]
                if edge.tgt in targets:
                    results.append(ReasoningPath(nodes=new_nodes, edges=new_edges, score=new_score))
                new_frontier.append((new_score, new_nodes, new_edges))
        new_frontier.sort(key=lambda item: item[0], reverse=True)
        frontier = new_frontier[: max(8, topk * 2)]
        if not frontier:
            break
    return results[:topk]


class GraphPathEncoder(nn.Module):
    def __init__(self, hidden_dim: int = 96, out_dim: int = 128) -> None:
        super().__init__()
        self.node_proj = nn.Linear(6, hidden_dim)
        self.rel_embedding = nn.Embedding(32, hidden_dim)
        self.message_proj = nn.Linear(hidden_dim * 2, hidden_dim)
        self.update_proj = nn.GRUCell(hidden_dim, hidden_dim)
        self.output_proj = nn.Linear(hidden_dim + 6, out_dim)
        self.rel_to_id = {rel: idx for idx, rel in enumerate(sorted(IMPORTANT_RELATIONS | {"relatedto"}), start=1)}

    def forward(self, retrieved: RetrievedSubgraph) -> torch.Tensor:
        device = next(self.parameters()).device
        if not retrieved.paths:
            return torch.zeros(1, self.output_proj.out_features, device=device)
        graph = retrieved.graph
        unique_nodes = list(dict.fromkeys(node_id for path in retrieved.paths for node_id in path.nodes))
        node_index = {node_id: idx for idx, node_id in enumerate(unique_nodes)}
        features = torch.zeros(len(unique_nodes), 6, device=device)
        degree = Counter()
        position = Counter()
        for path in retrieved.paths:
            for step, node_id in enumerate(path.nodes):
                position[node_id] += step
            for edge in path.edges:
                degree[edge.src] += 1
                degree[edge.tgt] += 1
        for node_id, idx in node_index.items():
            node = graph.nodes[node_id]
            features[idx, 0] = 1.0 if node.node_type == "context" else 0.0
            features[idx, 1] = 1.0 if node.node_type == "commonsense" else 0.0
            features[idx, 2] = min(1.0, len(simple_tokenize(node.label)) / 4.0)
            features[idx, 3] = min(1.0, degree[node_id] / 6.0)
            features[idx, 4] = 1.0 if any(node_id == path.nodes[-1] for path in retrieved.paths) else 0.0
            features[idx, 5] = min(1.0, position[node_id] / max(1, len(retrieved.paths) * 4))
        hidden = torch.tanh(self.node_proj(features))
        for path in retrieved.paths:
            for edge in path.edges:
                src_idx = node_index[edge.src]
                tgt_idx = node_index[edge.tgt]
                rel_id = self.rel_to_id.get(canonical_relation(edge.rel), 0)
                rel_vec = self.rel_embedding(torch.tensor([rel_id], device=device)).squeeze(0)
                message = torch.tanh(self.message_proj(torch.cat([hidden[src_idx], rel_vec], dim=-1)))
                hidden[tgt_idx] = self.update_proj(message, hidden[tgt_idx])
        pooled_nodes = hidden.mean(dim=0, keepdim=True)
        path_stats = torch.tensor(
            [[
                min(1.0, sum(path.hop_count for path in retrieved.paths) / max(1, len(retrieved.paths) * 4)),
                min(1.0, len(retrieved.path_nodes) / 8.0),
                min(1.0, lexical_overlap(" ".join(retrieved.path_nodes), " ".join(graph.nodes[n].label for n in unique_nodes))),
                1.0 if retrieved.hop_count > 0 else 0.0,
                min(1.0, len(unique_nodes) / 16.0),
                min(1.0, len(retrieved.paths) / 5.0),
            ]],
            dtype=torch.float32,
            device=device,
        )
        return torch.tanh(self.output_proj(torch.cat([pooled_nodes, path_stats], dim=-1)))


Writing /content/graphs_build.py


## `generator.py`

In [13]:
%%writefile /content/generator.py
from __future__ import annotations

from dataclasses import dataclass
import math
from typing import Dict, List, Optional

import torch
import torch.nn as nn

try:
    from transformers import BartForConditionalGeneration, BartTokenizer
except Exception:  # pragma: no cover
    BartForConditionalGeneration = None
    BartTokenizer = None


@dataclass
class ConditioningInputs:
    path_features: torch.Tensor
    skeleton_id: torch.Tensor
    form_id: torch.Tensor
    difficulty_id: torch.Tensor
    expressive_form: torch.Tensor
    expressive_mask: torch.Tensor


class ControllableGenerator(nn.Module):
    """
    Colab-safe BART generator.

    This keeps the paper-inspired conditioning signals, but avoids the brittle
    custom decoder-time control path that caused empty or one-token collapse.
    Conditioning is injected on the encoder side as learned control tokens.
    """

    def __init__(
        self,
        model_name: str = "facebook/bart-large",
        hidden_dim: Optional[int] = None,
        n_skeletons: int = 64,
        n_forms: int = 64,
        n_difficulty: int = 8,
    ) -> None:
        super().__init__()
        if BartTokenizer is None or BartForConditionalGeneration is None:
            raise ImportError("transformers is required for the generator")

        self.tokenizer = BartTokenizer.from_pretrained(model_name)
        try:
            self.model = BartForConditionalGeneration.from_pretrained(model_name, use_safetensors=True)
        except TypeError:
            self.model = BartForConditionalGeneration.from_pretrained(model_name)

        # Keep everything in fp32 for Colab stability.
        self.model = self.model.float()
        self.model.config.tie_word_embeddings = False
        self.model.generation_config.forced_bos_token_id = 0

        self.hidden_dim = hidden_dim or int(self.model.config.d_model)
        self.path_proj = nn.Sequential(nn.Linear(128, self.hidden_dim), nn.Tanh())
        self.skeleton_embedding = nn.Embedding(n_skeletons, self.hidden_dim)
        self.form_embedding = nn.Embedding(n_forms, self.hidden_dim)
        self.difficulty_embedding = nn.Embedding(n_difficulty, self.hidden_dim)
        self.gate = nn.Sequential(
            nn.Linear(self.hidden_dim * 4, self.hidden_dim),
            nn.Tanh(),
            nn.Linear(self.hidden_dim, self.hidden_dim),
        )
        self.control_norm = nn.LayerNorm(self.hidden_dim)

    def _encoder_embed_scale(self) -> float:
        encoder = self.model.model.encoder
        if hasattr(encoder, "embed_scale"):
            return float(encoder.embed_scale)
        if getattr(self.model.config, "scale_embedding", False):
            return math.sqrt(float(self.model.config.d_model))
        return 1.0

    def build_condition_vector(self, conditioning: ConditioningInputs) -> torch.Tensor:
        path_vec = self.path_proj(conditioning.path_features.float())
        sk_vec = self.skeleton_embedding(conditioning.skeleton_id)
        form_vec = self.form_embedding(conditioning.form_id)
        diff_vec = self.difficulty_embedding(conditioning.difficulty_id)
        return self.gate(torch.cat([path_vec, sk_vec, form_vec, diff_vec], dim=-1))

    def build_control_tokens(self, conditioning: ConditioningInputs) -> torch.Tensor:
        path_vec = self.path_proj(conditioning.path_features.float())
        sk_vec = self.skeleton_embedding(conditioning.skeleton_id)
        form_vec = self.form_embedding(conditioning.form_id)
        diff_vec = self.difficulty_embedding(conditioning.difficulty_id)
        fused = self.build_condition_vector(conditioning)
        stacked = torch.stack([fused, path_vec, sk_vec + diff_vec, form_vec], dim=1)
        return self.control_norm(stacked)

    def encode_inputs(
        self,
        contexts: List[str],
        answers: List[str],
        conditioning: ConditioningInputs,
        device: torch.device,
    ) -> Dict[str, torch.Tensor]:
        texts = [f"context: {context} answer: {answer}" for context, answer in zip(contexts, answers)]
        tokenized = self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        )
        tokenized = {key: value.to(device) for key, value in tokenized.items()}

        encoder = self.model.model.encoder
        token_embeds = encoder.embed_tokens(tokenized["input_ids"]).float() * self._encoder_embed_scale()
        control_tokens = self.build_control_tokens(conditioning).to(device=device, dtype=torch.float32)
        inputs_embeds = torch.cat([control_tokens, token_embeds], dim=1)

        cond_mask = torch.ones(
            (tokenized["attention_mask"].size(0), control_tokens.size(1)),
            dtype=tokenized["attention_mask"].dtype,
            device=device,
        )
        attention_mask = torch.cat([cond_mask, tokenized["attention_mask"]], dim=1)
        return {"inputs_embeds": inputs_embeds, "attention_mask": attention_mask}

    def forward(
        self,
        contexts: List[str],
        answers: List[str],
        questions: Optional[List[str]],
        conditioning: ConditioningInputs,
        device: torch.device,
    ):
        encoded = self.encode_inputs(contexts, answers, conditioning, device)
        labels = None
        if questions is not None:
            label_inputs = self.tokenizer(
                questions,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=64,
            )
            labels = label_inputs["input_ids"].to(device)
            labels[labels == self.tokenizer.pad_token_id] = -100
        return self.model(labels=labels, **encoded)

    @torch.no_grad()
    def generate(
        self,
        contexts: List[str],
        answers: List[str],
        conditioning: ConditioningInputs,
        device: torch.device,
        do_sample: bool = False,
        num_return_sequences: int = 1,
        max_length: int = 48,
        min_length: int = 8,
    ) -> List[str]:
        encoded = self.encode_inputs(contexts, answers, conditioning, device)
        encoder_outputs = self.model.model.encoder(
            inputs_embeds=encoded["inputs_embeds"],
            attention_mask=encoded["attention_mask"],
            return_dict=True,
        )
        outputs = self.model.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=encoded["attention_mask"],
            do_sample=do_sample,
            top_k=50 if do_sample else None,
            temperature=0.2,
            top_p=0.9 if do_sample else 1.0,
            num_beams=1,
            num_return_sequences=num_return_sequences,
            min_length=min_length,
            max_length=max_length,
            no_repeat_ngram_size=2,
            early_stopping=True,
        )
        decoded = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)
        return [text if text.strip() else "What happened?" for text in decoded]


def stack_conditioning(items: List[Dict], device: torch.device) -> ConditioningInputs:
    # The full expressive-form tensors are preserved for compatibility, even
    # though the stable generator version only uses the coarse form ID.
    max_form_len = max((len(item.get("sampled_form", [])) for item in items), default=0)
    form_tensor = []
    form_mask = []
    for item in items:
        form = list(item.get("sampled_form", []))
        padded = form + [0] * max(0, max_form_len - len(form))
        mask = [1] * len(form) + [0] * max(0, max_form_len - len(form))
        form_tensor.append(padded)
        form_mask.append(mask)
    return ConditioningInputs(
        path_features=torch.cat([item["path_features"] for item in items], dim=0).to(device),
        skeleton_id=torch.tensor([item["skeleton_id"] for item in items], dtype=torch.long, device=device),
        form_id=torch.tensor([item["form_id"] for item in items], dtype=torch.long, device=device),
        difficulty_id=torch.tensor([item["difficulty_level"] for item in items], dtype=torch.long, device=device),
        expressive_form=torch.tensor(form_tensor, dtype=torch.long, device=device) if max_form_len > 0 else torch.zeros((len(items), 0), dtype=torch.long, device=device),
        expressive_mask=torch.tensor(form_mask, dtype=torch.long, device=device) if max_form_len > 0 else torch.zeros((len(items), 0), dtype=torch.long, device=device),
    )


Writing /content/generator.py


## `evaluation.py`

In [14]:
%%writefile /content/evaluation.py
from __future__ import annotations

import math
from collections import Counter
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

from hsmm import HSMMArtifacts, HSMMTrainer
from skeletons import TraceExtractor

try:
    from bert_score import score as bert_score_fn
except Exception:  # pragma: no cover
    bert_score_fn = None


def tokenize(text: str) -> List[str]:
    return [tok for tok in text.lower().split() if tok]


def ngrams(tokens: Sequence[str], n: int) -> List[Tuple[str, ...]]:
    if n <= 0 or len(tokens) < n:
        return []
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def distinct_n(texts: Sequence[str], n: int = 2) -> float:
    grams = []
    for text in texts:
        grams.extend(ngrams(tokenize(text), n))
    if not grams:
        return 0.0
    return len(set(grams)) / len(grams)


def modified_precision(reference: str, prediction: str, n: int) -> float:
    ref_ngrams = Counter(ngrams(tokenize(reference), n))
    pred_ngrams = Counter(ngrams(tokenize(prediction), n))
    if not pred_ngrams:
        return 0.0
    overlap = sum((ref_ngrams & pred_ngrams).values())
    total = sum(pred_ngrams.values())
    return overlap / total if total else 0.0


def corpus_bleu(references: Sequence[str], predictions: Sequence[str], max_order: int = 4, smooth: float = 1e-9) -> float:
    if not references or not predictions:
        return 0.0
    matches = [0.0] * max_order
    totals = [0.0] * max_order
    ref_length = 0
    pred_length = 0
    for reference, prediction in zip(references, predictions):
        ref_tokens = tokenize(reference)
        pred_tokens = tokenize(prediction)
        ref_length += len(ref_tokens)
        pred_length += len(pred_tokens)
        for order in range(1, max_order + 1):
            ref_ngrams = Counter(ngrams(ref_tokens, order))
            pred_ngrams = Counter(ngrams(pred_tokens, order))
            matches[order - 1] += sum((ref_ngrams & pred_ngrams).values())
            totals[order - 1] += sum(pred_ngrams.values())
    if pred_length == 0:
        return 0.0
    precisions = [(matches[i] + smooth) / (totals[i] + smooth) for i in range(max_order)]
    brevity_penalty = 1.0 if pred_length > ref_length else math.exp(1.0 - (ref_length / pred_length))
    return brevity_penalty * math.exp(sum(math.log(max(p, smooth)) for p in precisions) / max_order)


def bleu_like(reference: str, prediction: str) -> float:
    return modified_precision(reference, prediction, 1)


def lcs_length(xs: Sequence[str], ys: Sequence[str]) -> int:
    if not xs or not ys:
        return 0
    dp = [0] * (len(ys) + 1)
    for x in xs:
        prev = 0
        for j, y in enumerate(ys, start=1):
            current = dp[j]
            if x == y:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = current
    return dp[-1]


def rouge_l_score(reference: str, prediction: str) -> float:
    ref_tokens = tokenize(reference)
    pred_tokens = tokenize(prediction)
    if not ref_tokens or not pred_tokens:
        return 0.0
    lcs = lcs_length(ref_tokens, pred_tokens)
    precision = lcs / len(pred_tokens)
    recall = lcs / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    beta = 1.2
    return ((1 + beta**2) * precision * recall) / (recall + beta**2 * precision)


def meteor_score(reference: str, prediction: str) -> float:
    ref_tokens = tokenize(reference)
    pred_tokens = tokenize(prediction)
    if not ref_tokens or not pred_tokens:
        return 0.0

    ref_positions: Dict[str, List[int]] = {}
    for idx, token in enumerate(ref_tokens):
        ref_positions.setdefault(token, []).append(idx)

    matches = []
    used_positions = set()
    for token in pred_tokens:
        positions = ref_positions.get(token, [])
        chosen = None
        for pos in positions:
            if pos not in used_positions:
                chosen = pos
                break
        if chosen is not None:
            used_positions.add(chosen)
            matches.append(chosen)

    match_count = len(matches)
    if match_count == 0:
        return 0.0

    precision = match_count / len(pred_tokens)
    recall = match_count / len(ref_tokens)
    f_mean = (10 * precision * recall) / (recall + 9 * precision) if (precision + recall) else 0.0

    chunks = 1
    for prev, current in zip(matches, matches[1:]):
        if current != prev + 1:
            chunks += 1
    penalty = 0.5 * ((chunks / match_count) ** 3)
    return (1 - penalty) * f_mean


def self_bleu(predictions: Sequence[str], order: int = 4) -> float:
    if len(predictions) <= 1:
        return 0.0
    scores = []
    for idx, prediction in enumerate(predictions):
        other_predictions = list(predictions[:idx]) + list(predictions[idx + 1 :])
        if not other_predictions:
            continue
        references = other_predictions if len(other_predictions) == 1 else [" ".join(tokenize(text)) for text in other_predictions]
        reference = max(references, key=lambda candidate: corpus_bleu([candidate], [prediction], max_order=order))
        scores.append(corpus_bleu([reference], [prediction], max_order=order))
    return sum(scores) / len(scores) if scores else 0.0


def maybe_compute_bertscore(
    references: Sequence[str],
    predictions: Sequence[str],
    lang: str = "en",
    model_type: Optional[str] = None,
) -> Dict[str, Optional[float]]:
    if not references or not predictions:
        return {
            "bertscore_precision": None,
            "bertscore_recall": None,
            "bertscore_f1": None,
        }
    if bert_score_fn is None:
        return {
            "bertscore_precision": None,
            "bertscore_recall": None,
            "bertscore_f1": None,
        }
    try:
        precision, recall, f1 = bert_score_fn(
            list(predictions),
            list(references),
            lang=lang,
            model_type=model_type,
            verbose=False,
        )
        return {
            "bertscore_precision": float(precision.mean().item()),
            "bertscore_recall": float(recall.mean().item()),
            "bertscore_f1": float(f1.mean().item()),
        }
    except Exception:
        return {
            "bertscore_precision": None,
            "bertscore_recall": None,
            "bertscore_f1": None,
        }


def per_sample_bertscore(
    references: Sequence[str],
    predictions: Sequence[str],
    lang: str = "en",
    model_type: Optional[str] = None,
) -> List[Optional[float]]:
    if not references or not predictions:
        return []
    if bert_score_fn is None:
        return [None for _ in predictions]
    try:
        _, _, f1 = bert_score_fn(
            list(predictions),
            list(references),
            lang=lang,
            model_type=model_type,
            verbose=False,
        )
        return [float(value.item()) for value in f1]
    except Exception:
        return [None for _ in predictions]


def per_sample_overlap_metrics(
    references: Sequence[str],
    predictions: Sequence[str],
) -> List[Dict[str, Optional[float]]]:
    bertscores = per_sample_bertscore(references, predictions)
    rows: List[Dict[str, Optional[float]]] = []
    for index, (reference, prediction) in enumerate(zip(references, predictions)):
        bertscore = bertscores[index] if index < len(bertscores) else None
        rows.append(
            {
                "bleu_4": corpus_bleu([reference], [prediction], max_order=4),
                "rouge_l": rouge_l_score(reference, prediction),
                "meteor": meteor_score(reference, prediction),
                "bertscore": bertscore,
            }
        )
    return rows


class RewardSuite:
    def __init__(self, hsmm_trainer: Optional[HSMMTrainer] = None, hsmm_artifacts: Optional[HSMMArtifacts] = None) -> None:
        self.trace_extractor = TraceExtractor()
        self.hsmm_trainer = hsmm_trainer
        self.hsmm_artifacts = hsmm_artifacts

    def compute(
        self,
        generated_question: str,
        context: str,
        answer: str,
        target_difficulty: int,
        expected_form: Optional[List[int]] = None,
        path_nodes: Optional[List[str]] = None,
    ) -> Dict[str, float]:
        rewards: Dict[str, float] = {}
        rewards["r1_answer_consistency"] = self._answer_consistency(generated_question, context, answer)
        rewards["r2_context_relevance"] = self._context_relevance(generated_question, context)
        rewards["r3_shortcut_resistance"] = self._shortcut_resistance(generated_question, context, answer)
        rewards["r4_hop_accuracy"] = self._hop_accuracy(generated_question, target_difficulty)
        rewards["r5_commonsense_coverage"] = self._commonsense_coverage(context, path_nodes or [])
        rewards["r6_form_match"] = self._form_match(generated_question, expected_form or [])
        rewards["r7_fluency"] = self._fluency(generated_question)
        weights = {
            "r1_answer_consistency": 0.2,
            "r2_context_relevance": 0.1,
            "r3_shortcut_resistance": 0.2,
            "r4_hop_accuracy": 0.1,
            "r5_commonsense_coverage": 0.2,
            "r6_form_match": 0.1,
            "r7_fluency": 0.1,
        }
        rewards["total"] = sum(weights[key] * rewards[key] for key in weights)
        return rewards

    def _answer_consistency(self, question: str, context: str, answer: str) -> float:
        q_tokens = set(tokenize(question))
        c_tokens = set(tokenize(context))
        a_tokens = set(tokenize(answer))
        if not q_tokens:
            return 0.0
        coverage = len((q_tokens | a_tokens) & c_tokens) / len(q_tokens | a_tokens or {"_"})
        answer_hint = 1.0 if any(token in tokenize(context) for token in a_tokens) else 0.0
        return min(1.0, 0.6 * coverage + 0.4 * answer_hint)

    def _context_relevance(self, question: str, context: str) -> float:
        q_tokens = set(tokenize(question))
        c_tokens = set(tokenize(context))
        if not q_tokens:
            return 0.0
        return len(q_tokens & c_tokens) / len(q_tokens)

    def _shortcut_resistance(self, question: str, context: str, answer: str) -> float:
        q_tokens = set(tokenize(question))
        if not q_tokens:
            return 0.0
        max_overlap = 0.0
        for sentence in context.split("."):
            s_tokens = set(tokenize(sentence))
            if not s_tokens:
                continue
            max_overlap = max(max_overlap, len(q_tokens & s_tokens) / len(q_tokens))
        answer_tokens = set(tokenize(answer))
        answer_overlap = len(answer_tokens & q_tokens) / len(answer_tokens) if answer_tokens else 0.0
        return max(0.0, 1.0 - (0.7 * max_overlap + 0.3 * answer_overlap))

    def _hop_accuracy(self, question: str, target_difficulty: int) -> float:
        trace = self.trace_extractor.extract_from_question(question)
        diff = abs(trace.hop_count - target_difficulty)
        return 1.0 / math.log2(2.0 + diff)

    def _commonsense_coverage(self, context: str, path_nodes: Sequence[str]) -> float:
        context_tokens = set(tokenize(context))
        if not path_nodes:
            return 0.0
        unseen = 0
        for node in path_nodes:
            node_tokens = set(tokenize(node))
            if node_tokens and not node_tokens.issubset(context_tokens):
                unseen += 1
        return unseen / len(path_nodes)

    def _form_match(self, question: str, expected_form: Sequence[int]) -> float:
        if not expected_form:
            return 0.5
        if self.hsmm_trainer is None or self.hsmm_artifacts is None:
            return 0.5
        segments = self.hsmm_trainer.segment(question, self.hsmm_artifacts)
        predicted = {state for state, _, _ in segments}
        expected = set(expected_form)
        if not predicted and not expected:
            return 1.0
        if not predicted or not expected:
            return 0.0
        return len(predicted & expected) / len(predicted | expected)

    def _fluency(self, question: str) -> float:
        toks = tokenize(question)
        if not toks:
            return 0.0
        repeat_penalty = 1.0 - (len(toks) - len(set(toks))) / max(1, len(toks))
        length_bonus = 1.0 if 4 <= len(toks) <= 20 else max(0.2, 1.0 - abs(len(toks) - 12) / 20.0)
        punctuation_bonus = 1.0 if question.strip().endswith("?") else 0.7
        return max(0.0, min(1.0, 0.4 * repeat_penalty + 0.4 * length_bonus + 0.2 * punctuation_bonus))


def evaluate_predictions(records: Iterable[Dict], reward_suite: Optional[RewardSuite] = None) -> Dict[str, Optional[float]]:
    records = list(records)
    if not records:
        return {"count": 0}

    references = [record.get("reference_question", "") for record in records]
    predictions = [record["generated_question"] for record in records]

    bleu_like_scores = []
    rouge_l_scores = []
    meteor_scores = []
    rewards = []
    hop_match = []
    lengths = []

    for record, prediction, reference in zip(records, predictions, references):
        bleu_like_scores.append(bleu_like(reference, prediction))
        rouge_l_scores.append(rouge_l_score(reference, prediction))
        meteor_scores.append(meteor_score(reference, prediction))
        hop_match.append(1.0 if record.get("predicted_hop") == record.get("target_difficulty") else 0.0)
        lengths.append(len(tokenize(prediction)))
        if reward_suite is not None:
            rewards.append(
                reward_suite.compute(
                    generated_question=prediction,
                    context=record["context"],
                    answer=record["answer"],
                    target_difficulty=record.get("target_difficulty", 2),
                    expected_form=record.get("expected_form", []),
                    path_nodes=record.get("path_nodes", []),
                )["total"]
            )

    metrics: Dict[str, Optional[float]] = {
        "count": len(records),
        "avg_bleu_like": sum(bleu_like_scores) / len(bleu_like_scores),
        "bleu_1": corpus_bleu(references, predictions, max_order=1),
        "bleu_2": corpus_bleu(references, predictions, max_order=2),
        "bleu_3": corpus_bleu(references, predictions, max_order=3),
        "bleu_4": corpus_bleu(references, predictions, max_order=4),
        "meteor": sum(meteor_scores) / len(meteor_scores),
        "rouge_l": sum(rouge_l_scores) / len(rouge_l_scores),
        "avg_reward": sum(rewards) / len(rewards) if rewards else 0.0,
        "difficulty_accuracy": sum(hop_match) / len(hop_match),
        "distinct_1": distinct_n(predictions, 1),
        "distinct_2": distinct_n(predictions, 2),
        "distinct_3": distinct_n(predictions, 3),
        "distinct_4": distinct_n(predictions, 4),
        "self_bleu_4": self_bleu(predictions, order=4),
        "avg_generated_length": sum(lengths) / len(lengths),
    }
    metrics.update(maybe_compute_bertscore(references, predictions))
    return metrics


Writing /content/evaluation.py


## `main.py`

In [15]:
%%writefile /content/main.py
from __future__ import annotations

import argparse
import csv
import json
import os
import random
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from typing import Dict, List, Sequence

import torch

from evaluation import RewardSuite, evaluate_predictions
from generator import ControllableGenerator, stack_conditioning
from graphs_build import (
    AMRProxyParser,
    CompositeKBBackend,
    GraphPathEncoder,
    build_context_graph,
    build_inferential_graph,
    extract_entities,
    prune_inferential_graph,
    retrieve_reasoning_subgraph,
)
from hsmm import HSMMTrainer
from skeletons import DifficultyController, SkeletonMiner, TraceExtractor, form_to_id, skeleton_to_id


@dataclass
class QGSample:
    sample_id: str
    context: str
    question: str
    answer: str
    difficulty_level: int


def resolve_data_path(data_dir: str, candidates: Sequence[str]) -> str:
    script_dir = os.path.dirname(os.path.abspath(__file__))
    search_roots: List[str] = []
    for root in [data_dir, os.path.join(script_dir, data_dir), "dataset", os.path.join(script_dir, "dataset")]:
        normalized = os.path.normpath(root)
        if normalized not in search_roots:
            search_roots.append(normalized)
    for root in search_roots:
        for filename in candidates:
            path = os.path.join(root, filename)
            if os.path.exists(path):
                return path
    searched = [os.path.join(root, filename) for root in search_roots for filename in candidates]
    raise FileNotFoundError(f"Could not find any dataset file. Tried: {searched}")


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def estimate_difficulty(question: str) -> int:
    q = question.lower()
    difficulty = 1
    if any(token in q for token in ["why", "because", "result", "effect"]):
        difficulty += 1
    if any(token in q for token in ["after", "before", "then", "next", "finally"]):
        difficulty += 1
    if any(token in q for token in ["purpose", "in order", "how"]):
        difficulty += 1
    return max(1, min(4, difficulty))


def load_cosmos_samples(data_dir: str, max_train: int = 200, max_dev: int = 50) -> Dict[str, List[QGSample]]:
    legacy_path = resolve_data_path(data_dir, ["Cosmostrain.csv", "train.csv"])
    rows = []
    with open(legacy_path, "r", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        for idx, row in enumerate(reader):
            label = int(row["label"])
            answer = row[f"answer{label}"]
            rows.append(
                QGSample(
                    sample_id=f"cosmos_{idx}",
                    context=row["context"],
                    question=row["question"],
                    answer=answer,
                    difficulty_level=estimate_difficulty(row["question"]),
                )
            )
    return {
        "train": rows[:max_train],
        "dev": rows[max_train:max_train + max_dev],
    }


def load_mcscript_samples(data_dir: str, max_train: int = 200, max_dev: int = 50) -> Dict[str, List[QGSample]]:
    path = resolve_data_path(data_dir, ["MCScript_train.xml", "train-data.xml"])
    tree = ET.parse(path)
    root = tree.getroot()
    samples = []
    idx = 0
    for instance in root.findall(".//instance"):
        context = instance.findtext("text", default="") or instance.get("text", "")
        for question in instance.findall(".//question"):
            q_text = question.findtext("text", default="") or question.get("text", "")
            answer = ""
            for ans in question.findall(".//answer"):
                if str(ans.get("correct", "")).strip().lower() in {"1", "true", "yes"}:
                    answer = ans.findtext("text", default="") or ans.get("text", "")
                    break
            if context and q_text and answer:
                samples.append(
                    QGSample(
                        sample_id=f"mcscript_{idx}",
                        context=context,
                        question=q_text,
                        answer=answer,
                        difficulty_level=2,
                    )
                )
                idx += 1
    return {
        "train": samples[:max_train],
        "dev": samples[max_train:max_train + max_dev],
    }


def _summarize_kqapro_program(program: Sequence[Dict], max_steps: int = 6) -> str:
    steps: List[str] = []
    for step in list(program)[:max_steps]:
        if not isinstance(step, dict):
            continue
        function = step.get("function", "unknown")
        inputs = ", ".join(str(item) for item in step.get("inputs", []))
        steps.append(f"{function}({inputs})" if inputs else str(function))
    return " -> ".join(steps)


def load_kqapro_samples(data_dir: str, max_train: int = 200, max_dev: int = 50) -> Dict[str, List[QGSample]]:
    path = resolve_data_path(data_dir, ["KQAPro_train.json"])
    with open(path, "r", encoding="utf-8") as handle:
        payload = json.load(handle)

    samples: List[QGSample] = []
    for idx, item in enumerate(payload):
        question = str(item.get("question", "")).strip()
        answer = str(item.get("answer", "")).strip()
        if not question or not answer:
            continue
        choices = [str(choice).strip() for choice in item.get("choices", []) if str(choice).strip()]
        program = _summarize_kqapro_program(item.get("program", []))
        context_parts = ["dataset: kqapro"]
        if choices:
            context_parts.append("choices: " + "; ".join(choices[:10]))
        if program:
            context_parts.append("program: " + program)
        context = " | ".join(context_parts)
        samples.append(
            QGSample(
                sample_id=f"kqapro_{idx}",
                context=context,
                question=question,
                answer=answer,
                difficulty_level=estimate_difficulty(question),
            )
        )
    return {
        "train": samples[:max_train],
        "dev": samples[max_train:max_train + max_dev],
    }


def _summarize_grail_graph(graph_query: Dict, max_nodes: int = 6, max_edges: int = 6) -> str:
    nodes = []
    for node in list(graph_query.get("nodes", []))[:max_nodes]:
        label = node.get("friendly_name") or node.get("class") or node.get("id")
        if label:
            nodes.append(str(label))
    edges = []
    for edge in list(graph_query.get("edges", []))[:max_edges]:
        relation = edge.get("friendly_name") or edge.get("relation")
        if relation:
            edges.append(str(relation))
    parts = []
    if nodes:
        parts.append("nodes: " + "; ".join(nodes))
    if edges:
        parts.append("relations: " + "; ".join(edges))
    return " | ".join(parts)


def load_grailqa_samples(data_dir: str, max_train: int = 200, max_dev: int = 50) -> Dict[str, List[QGSample]]:
    path = resolve_data_path(data_dir, ["grailqa_v1.0_train.json"])
    with open(path, "r", encoding="utf-8") as handle:
        payload = json.load(handle)

    samples: List[QGSample] = []
    for idx, item in enumerate(payload):
        question = str(item.get("question", "")).strip()
        answers = item.get("answer", [])
        answer_names = [str(ans.get("entity_name", "")).strip() for ans in answers if str(ans.get("entity_name", "")).strip()]
        answer = "; ".join(answer_names[:5])
        if not question or not answer:
            continue
        domains = [str(domain).strip() for domain in item.get("domains", []) if str(domain).strip()]
        graph_summary = _summarize_grail_graph(item.get("graph_query", {}))
        context_parts = ["dataset: grailqa"]
        if domains:
            context_parts.append("domains: " + "; ".join(domains[:5]))
        if graph_summary:
            context_parts.append(graph_summary)
        context = " | ".join(context_parts)
        samples.append(
            QGSample(
                sample_id=f"grailqa_{idx}",
                context=context,
                question=question,
                answer=answer,
                difficulty_level=estimate_difficulty(question),
            )
        )
    return {
        "train": samples[:max_train],
        "dev": samples[max_train:max_train + max_dev],
    }


def load_unlabeled_questions(data_dir: str, max_questions: int = 3000) -> List[str]:
    questions: List[str] = []
    for filename in ["KQAPro_train.json", "train.json", "grailqa_v1.0_train.json"]:
        try:
            path = resolve_data_path(data_dir, [filename])
        except FileNotFoundError:
            continue
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
        iterable = payload.values() if isinstance(payload, dict) else payload
        for item in iterable:
            question = item.get("question") if isinstance(item, dict) else None
            if question:
                questions.append(question)
            if len(questions) >= max_questions:
                return questions
    return questions


def build_conditioned_example(
    sample: QGSample,
    parser: AMRProxyParser,
    kb_backend: CompositeKBBackend,
    path_encoder: GraphPathEncoder,
    difficulty_controller: DifficultyController,
    hsmm_trainer: HSMMTrainer,
    hsmm_artifacts,
    trace_extractor: TraceExtractor,
) -> Dict:
    question_trace = trace_extractor.extract_from_text(sample.context, sample.question, sample.answer)
    amr = parser.parse(sample.context)
    context_graph = build_context_graph(amr)
    concepts = extract_entities(sample.context, sample.answer)
    triples = kb_backend.retrieve(concepts)
    inferential_graph = prune_inferential_graph(
        build_inferential_graph(context_graph, triples),
        answer_text=sample.answer,
    )

    skeleton = difficulty_controller.select(
        sample.difficulty_level,
        sample.question,
        preferred_relations=question_trace.relations,
        preferred_type=SkeletonMiner()._infer_type(question_trace.relations),
    )
    retrieved = retrieve_reasoning_subgraph(
        inferential_graph,
        answer_text=sample.answer,
        skeleton=skeleton,
        difficulty=sample.difficulty_level,
        topk=5,
    )
    path_features = path_encoder(retrieved)
    if retrieved.paths:
        path_trace = trace_extractor.extract_from_path(retrieved.paths[0], inferential_graph)
        predicted_hop = path_trace.hop_count
    else:
        path_trace = question_trace
        predicted_hop = path_trace.hop_count

    form_segments = hsmm_trainer.segment(sample.question, hsmm_artifacts)
    expected_form = [state for state, _, _ in form_segments]
    form_signature = getattr(skeleton, "reasoning_type", "") or SkeletonMiner()._infer_type(path_trace.relations)
    sampled_form = hsmm_artifacts.pool.sample(
        preferred_length=max(1, sample.difficulty_level),
        signature=form_signature,
    )
    return {
        "sample": sample,
        "path_features": path_features.detach(),
        "skeleton_id": skeleton_to_id(skeleton),
        "form_id": form_to_id(sampled_form),
        "sampled_form": sampled_form,
        "difficulty_level": sample.difficulty_level,
        "expected_form": expected_form,
        "path_nodes": retrieved.path_nodes,
        "predicted_hop": predicted_hop,
        "reasoning_type": form_signature,
    }


def pretrain(
    model: ControllableGenerator,
    examples: Sequence[Dict],
    device: torch.device,
    epochs: int = 1,
    batch_size: int = 4,
    lr: float = 1e-5,
) -> None:
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    for _ in range(epochs):
        order = list(examples)
        random.shuffle(order)
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            conditioning = stack_conditioning(batch, device)
            outputs = model(
                contexts=[item["sample"].context for item in batch],
                answers=[item["sample"].answer for item in batch],
                questions=[item["sample"].question for item in batch],
                conditioning=conditioning,
                device=device,
            )
            optimizer.zero_grad()
            outputs.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()


def generate_records(
    model: ControllableGenerator,
    examples: Sequence[Dict],
    device: torch.device,
    limit: int = 20,
    reward_suite: RewardSuite | None = None,
    num_candidates: int = 4,
) -> List[Dict]:
    records = []
    model.eval()
    for item in examples[:limit]:
        conditioning = stack_conditioning([item], device)
        greedy_candidates = model.generate(
            contexts=[item["sample"].context],
            answers=[item["sample"].answer],
            conditioning=conditioning,
            device=device,
            do_sample=False,
            num_return_sequences=1,
        )
        sampled_candidates = model.generate(
            contexts=[item["sample"].context],
            answers=[item["sample"].answer],
            conditioning=conditioning,
            device=device,
            do_sample=True,
            num_return_sequences=max(1, num_candidates),
        )
        candidates = []
        for text in greedy_candidates + sampled_candidates:
            normalized = " ".join((text or "").strip().split())
            if normalized and normalized not in candidates:
                candidates.append(normalized)
        if not candidates:
            candidates = ["What?"]

        if reward_suite is None:
            question = candidates[0]
        else:
            scored = [
                (
                    reward_suite.compute(
                        generated_question=candidate,
                        context=item["sample"].context,
                        answer=item["sample"].answer,
                        target_difficulty=item["difficulty_level"],
                        expected_form=item.get("expected_form", []),
                        path_nodes=item.get("path_nodes", []),
                    )["total"],
                    candidate,
                )
                for candidate in candidates
            ]
            scored.sort(key=lambda pair: pair[0], reverse=True)
            question = scored[0][1]
        records.append(
            {
                "context": item["sample"].context,
                "answer": item["sample"].answer,
                "reference_question": item["sample"].question,
                "generated_question": question,
                "target_difficulty": item["difficulty_level"],
                "predicted_hop": item["predicted_hop"],
                "expected_form": item["expected_form"],
                "path_nodes": item["path_nodes"],
            }
        )
    return records


def main() -> None:
    parser = argparse.ArgumentParser(description="Flowchart-faithful commonsense QG pipeline")
    parser.add_argument("--dataset", choices=["cosmos", "mcscript"], default="cosmos")
    parser.add_argument("--data_dir", default="dataset")
    parser.add_argument("--output_dir", default="outputs/flowchart_dcqg")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--train_limit", type=int, default=64)
    parser.add_argument("--dev_limit", type=int, default=16)
    parser.add_argument("--pretrain_epochs", type=int, default=3)
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)
    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if args.dataset == "cosmos":
        split = load_cosmos_samples(args.data_dir, max_train=args.train_limit, max_dev=args.dev_limit)
    else:
        split = load_mcscript_samples(args.data_dir, max_train=args.train_limit, max_dev=args.dev_limit)

    unlabeled_questions = load_unlabeled_questions(args.data_dir)
    unlabeled_questions.extend(sample.question for sample in split["train"])
    unlabeled_signatures = [
        TraceExtractor().extract_from_question(question).relations
        for question in unlabeled_questions
    ]
    unlabeled_signatures = [
        SkeletonMiner()._infer_type(relations)
        for relations in unlabeled_signatures
    ]

    skeleton_bank = SkeletonMiner().learn_from_questions(unlabeled_questions, min_support=2, max_hops=4)
    difficulty_controller = DifficultyController(skeleton_bank)
    hsmm_trainer = HSMMTrainer(device=str(device))
    hsmm_limit = min(2000, len(unlabeled_questions))
    hsmm_artifacts = hsmm_trainer.fit(
        unlabeled_questions[:hsmm_limit],
        epochs=2,
        lr=1e-3,
        signatures=unlabeled_signatures[:hsmm_limit],
    )

    with open(os.path.join(args.output_dir, "skeleton_bank.json"), "w", encoding="utf-8") as handle:
        json.dump(
            [
                {
                    "id": skeleton.id,
                    "reasoning_type": skeleton.reasoning_type,
                    "size": skeleton.size,
                    "support": skeleton.support,
                    "relations": [edge.relation for edge in skeleton.structure],
                }
                for skeleton in skeleton_bank
            ],
            handle,
            indent=2,
        )
    with open(os.path.join(args.output_dir, "hsmm_metadata.json"), "w", encoding="utf-8") as handle:
        json.dump(
            {
                "vocab_size": len(hsmm_artifacts.vocab),
                "num_forms": len(hsmm_artifacts.pool.forms),
                "sample_forms": hsmm_artifacts.pool.forms[:20],
            },
            handle,
            indent=2,
        )

    amr_parser = AMRProxyParser()
    kb_backend = CompositeKBBackend()
    path_encoder = GraphPathEncoder().to(device)
    trace_extractor = TraceExtractor()

    train_examples = [
        build_conditioned_example(
            sample,
            parser=amr_parser,
            kb_backend=kb_backend,
            path_encoder=path_encoder,
            difficulty_controller=difficulty_controller,
            hsmm_trainer=hsmm_trainer,
            hsmm_artifacts=hsmm_artifacts,
            trace_extractor=trace_extractor,
        )
        for sample in split["train"]
    ]
    dev_examples = [
        build_conditioned_example(
            sample,
            parser=amr_parser,
            kb_backend=kb_backend,
            path_encoder=path_encoder,
            difficulty_controller=difficulty_controller,
            hsmm_trainer=hsmm_trainer,
            hsmm_artifacts=hsmm_artifacts,
            trace_extractor=trace_extractor,
        )
        for sample in split["dev"]
    ]

    # Legacy BART initialization retained below as a comment.
    # model = ControllableGenerator().to(device)
    model = ControllableGenerator(model_name=os.getenv("GEMINI_MODEL", "gemini-2.0-flash")).to(device)
    pretrain(model, train_examples, device=device, epochs=max(1, args.pretrain_epochs), batch_size=2, lr=1e-5)

    reward_suite = RewardSuite(hsmm_trainer=hsmm_trainer, hsmm_artifacts=hsmm_artifacts)

    dev_records = generate_records(model, dev_examples, device=device, limit=len(dev_examples))
    metrics = evaluate_predictions(dev_records, reward_suite=reward_suite)

    with open(os.path.join(args.output_dir, "dev_predictions.jsonl"), "w", encoding="utf-8") as handle:
        for record in dev_records:
            handle.write(json.dumps(record, ensure_ascii=True) + "\n")
    with open(os.path.join(args.output_dir, "metrics.json"), "w", encoding="utf-8") as handle:
        json.dump(metrics, handle, indent=2)
    torch.save(model.state_dict(), os.path.join(args.output_dir, "generator.pt"))
    torch.save(path_encoder.state_dict(), os.path.join(args.output_dir, "graph_encoder.pt"))
    torch.save(hsmm_artifacts.model.state_dict(), os.path.join(args.output_dir, "hsmm.pt"))

    print(json.dumps(metrics, indent=2))


if __name__ == "__main__":
    main()


Writing /content/main.py


In [16]:
%%writefile /content/export_four_dataset_workbook.py
from __future__ import annotations

import argparse
import json
import os
import zipfile
import xml.etree.ElementTree as ET
from html import escape
from typing import Dict, Iterable, List

import torch
import pandas as pd  # type: ignore

from evaluation import per_sample_overlap_metrics
from generator import ControllableGenerator
from graphs_build import AMRProxyParser, CompositeKBBackend, GraphPathEncoder
from hsmm import HSMMTrainer
from main import (
    build_conditioned_example,
    generate_records,
    load_cosmos_samples,
    load_grailqa_samples,
    load_kqapro_samples,
    load_mcscript_samples,
    load_unlabeled_questions,
    pretrain,
    set_seed,
)
from skeletons import DifficultyController, SkeletonMiner, TraceExtractor
from evaluation import RewardSuite


DATASET_LOADERS = {
    "CosmosQA": load_cosmos_samples,
    "GrailQA": load_grailqa_samples,
    "MCscript": load_mcscript_samples,
    "KQApro": load_kqapro_samples,
}


def _column_name(index: int) -> str:
    name = ""
    while index >= 0:
        name = chr(ord("A") + (index % 26)) + name
        index = (index // 26) - 1
    return name


def _col_to_index(col: str) -> int:
    idx = 0
    for ch in col:
        idx = idx * 26 + (ord(ch.upper()) - ord("A") + 1)
    return idx - 1


def _parse_sheet_xml(sheet_xml: str, shared_strings: List[str]) -> List[List[str]]:
    if not sheet_xml:
        return []
    ns = {"ns": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    root = ET.fromstring(sheet_xml)
    rows: List[List[str]] = []
    for row in root.findall("ns:sheetData/ns:row", ns):
        cells: Dict[int, str] = {}
        max_col = -1
        for c in row.findall("ns:c", ns):
            ref = c.get("r", "")
            col_letters = "".join(ch for ch in ref if ch.isalpha())
            if not col_letters:
                continue
            col_idx = _col_to_index(col_letters)
            max_col = max(max_col, col_idx)
            cell_type = c.get("t")
            value = ""
            if cell_type == "s":
                v = c.find("ns:v", ns)
                if v is not None and v.text is not None:
                    try:
                        idx = int(v.text)
                        value = shared_strings[idx] if 0 <= idx < len(shared_strings) else ""
                    except ValueError:
                        value = ""
            elif cell_type == "inlineStr":
                t = c.find("ns:is/ns:t", ns)
                value = t.text if t is not None and t.text is not None else ""
            else:
                v = c.find("ns:v", ns)
                value = v.text if v is not None and v.text is not None else ""
            cells[col_idx] = value
        if max_col >= 0:
            row_values = [""] * (max_col + 1)
            for idx, value in cells.items():
                row_values[idx] = value
            rows.append(row_values)
    return rows


def read_xlsx(workbook_path: str) -> Dict[str, List[Dict[str, object]]]:
    if not os.path.exists(workbook_path):
        return {}
    sheets: Dict[str, List[Dict[str, object]]] = {}
    xls = pd.ExcelFile(workbook_path)
    for name in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=name)
        sheets[name] = df.to_dict(orient="records")
    return sheets


def _xlsx_sheet_xml(rows: List[Dict[str, object]], headers: List[str], string_index: Dict[str, int]) -> str:
    xml_rows: List[str] = []
    all_rows: List[Iterable[object]] = [headers]
    all_rows.extend([[row.get(header, "") for header in headers] for row in rows])
    for row_idx, row in enumerate(all_rows, start=1):
        cells: List[str] = []
        for col_idx, value in enumerate(row):
            cell_ref = f"{_column_name(col_idx)}{row_idx}"
            if value is None:
                idx = string_index.setdefault("", len(string_index))
                cells.append(f'<c r="{cell_ref}" t="s"><v>{idx}</v></c>')
            elif isinstance(value, (int, float)) and not isinstance(value, bool):
                cells.append(f'<c r="{cell_ref}"><v>{value}</v></c>')
            else:
                text = escape(str(value))
                idx = string_index.setdefault(text, len(string_index))
                cells.append(f'<c r="{cell_ref}" t="s"><v>{idx}</v></c>')
        xml_rows.append(f'<row r="{row_idx}">{"".join(cells)}</row>')
    return (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
        '<worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">'
        "<sheetData>"
        + "".join(xml_rows)
        + "</sheetData></worksheet>"
    )


def write_xlsx(workbook_path: str, sheets: Dict[str, List[Dict[str, object]]], headers: List[str]) -> None:
    os.makedirs(os.path.dirname(workbook_path), exist_ok=True)
    sheet_names = list(sheets.keys())
    string_index: Dict[str, int] = {}
    for name in sheet_names:
        for row in [headers] + [[row.get(header, "") for header in headers] for row in sheets[name]]:
            for value in row:
                if value is None or isinstance(value, (int, float)) and not isinstance(value, bool):
                    continue
                text = escape(str(value))
                if text not in string_index:
                    string_index[text] = len(string_index)
    if "" not in string_index:
        string_index[""] = len(string_index)
    content_types = [
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>',
        '<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types">',
        '<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>',
        '<Default Extension="xml" ContentType="application/xml"/>',
        '<Override PartName="/xl/workbook.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/>',
        '<Override PartName="/xl/styles.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.styles+xml"/>',
        '<Override PartName="/xl/sharedStrings.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sharedStrings+xml"/>',
    ]
    for idx in range(1, len(sheet_names) + 1):
        content_types.append(
            f'<Override PartName="/xl/worksheets/sheet{idx}.xml" '
            'ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/>'
        )
    content_types.append("</Types>")

    workbook = [
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>',
        '<workbook xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" '
        'xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships"><sheets>',
    ]
    for idx, name in enumerate(sheet_names, start=1):
        workbook.append(f'<sheet name="{escape(name[:31])}" sheetId="{idx}" r:id="rId{idx}"/>')
    workbook.append("</sheets></workbook>")

    workbook_rels = [
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>',
        '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">',
    ]
    for idx in range(1, len(sheet_names) + 1):
        workbook_rels.append(
            f'<Relationship Id="rId{idx}" '
            'Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" '
            f'Target="worksheets/sheet{idx}.xml"/>'
        )
    workbook_rels.append(
        '<Relationship Id="rId999" '
        'Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/styles" '
        'Target="styles.xml"/>'
    )
    workbook_rels.append("</Relationships>")

    root_rels = (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
        '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">'
        '<Relationship Id="rId1" '
        'Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" '
        'Target="xl/workbook.xml"/>'
        "</Relationships>"
    )
    styles = (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
        '<styleSheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">'
        '<fonts count="1"><font><sz val="11"/><name val="Calibri"/></font></fonts>'
        '<fills count="1"><fill><patternFill patternType="none"/></fill></fills>'
        '<borders count="1"><border/></borders>'
        '<cellStyleXfs count="1"><xf numFmtId="0" fontId="0" fillId="0" borderId="0"/></cellStyleXfs>'
        '<cellXfs count="1"><xf numFmtId="0" fontId="0" fillId="0" borderId="0" xfId="0"/></cellXfs>'
        '<cellStyles count="1"><cellStyle name="Normal" xfId="0" builtinId="0"/></cellStyles>'
        "</styleSheet>"
    )

    with zipfile.ZipFile(workbook_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("[Content_Types].xml", "".join(content_types))
        archive.writestr("_rels/.rels", root_rels)
        archive.writestr("xl/workbook.xml", "".join(workbook))
        archive.writestr("xl/_rels/workbook.xml.rels", "".join(workbook_rels))
        archive.writestr("xl/styles.xml", styles)
        shared_strings = ['<?xml version="1.0" encoding="UTF-8" standalone="yes"?>',
                          '<sst xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" '
                          f'count="{len(string_index)}" uniqueCount="{len(string_index)}">']
        index_to_string = [""] * len(string_index)
        for text, idx in string_index.items():
            index_to_string[idx] = text
        for text in index_to_string:
            shared_strings.append(f'<si><t xml:space="preserve">{text}</t></si>')
        shared_strings.append("</sst>")
        archive.writestr("xl/sharedStrings.xml", "".join(shared_strings))
        for idx, name in enumerate(sheet_names, start=1):
            archive.writestr(f"xl/worksheets/sheet{idx}.xml", _xlsx_sheet_xml(sheets[name], headers, string_index))


def write_csv_outputs(output_dir: str, sheets: Dict[str, List[Dict[str, object]]], headers: List[str]) -> None:
    os.makedirs(output_dir, exist_ok=True)
    for name, rows in sheets.items():
        csv_path = os.path.join(output_dir, f"{name}.csv")
        with open(csv_path, "w", encoding="utf-8") as handle:
            handle.write(",".join(headers) + "\n")
            for row in rows:
                values = []
                for header in headers:
                    value = row.get(header, "")
                    text = str(value).replace('"', '""')
                    if "," in text or "\n" in text:
                        text = f"\"{text}\""
                    values.append(text)
                handle.write(",".join(values) + "\n")


def write_xlsx_pandas(workbook_path: str, sheets: Dict[str, List[Dict[str, object]]], headers: List[str]) -> None:
    os.makedirs(os.path.dirname(workbook_path), exist_ok=True)
    with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
        for name, rows in sheets.items():
            df = pd.DataFrame(rows, columns=headers)
            df.to_excel(writer, sheet_name=name[:31], index=False)


def build_sheet_rows(records: List[Dict[str, object]]) -> List[Dict[str, object]]:
    references = [str(record.get("reference_question", "")) for record in records]
    predictions = [str(record.get("generated_question", "")) for record in records]
    metric_rows = per_sample_overlap_metrics(references, predictions)

    rows: List[Dict[str, object]] = []
    for record, metrics in zip(records, metric_rows):
        rows.append(
            {
                "sample_id": record.get("sample_id", ""),
                "context": record.get("context", ""),
                "answer": record.get("answer", ""),
                "reference_question": record.get("reference_question", ""),
                "generated_question": record.get("generated_question", ""),
                "target_difficulty": record.get("target_difficulty", ""),
                "predicted_hop": record.get("predicted_hop", ""),
                "bleu_4": metrics["bleu_4"],
                "rouge_l": metrics["rouge_l"],
                "meteor": metrics["meteor"],
                "bertscore": metrics["bertscore"],
            }
        )
    return rows


def fit_shared_structural_models(
    data_dir: str,
    selected_samples: Dict[str, List],
    device: torch.device,
):
    unlabeled_questions = load_unlabeled_questions(data_dir, max_questions=3000)
    all_samples = [sample for samples in selected_samples.values() for sample in samples]
    unlabeled_questions.extend(sample.question for sample in all_samples)
    trace_extractor = TraceExtractor()
    unlabeled_signatures = [
        SkeletonMiner()._infer_type(trace_extractor.extract_from_question(question).relations)
        for question in unlabeled_questions
    ]

    skeleton_bank = SkeletonMiner().learn_from_questions(unlabeled_questions, min_support=2, max_hops=4)
    difficulty_controller = DifficultyController(skeleton_bank)
    hsmm_trainer = HSMMTrainer(device=str(device))
    hsmm_limit = min(2000, len(unlabeled_questions))
    hsmm_artifacts = hsmm_trainer.fit(
        unlabeled_questions[:hsmm_limit],
        epochs=2,
        lr=1e-3,
        signatures=unlabeled_signatures[:hsmm_limit],
    )
    return trace_extractor, difficulty_controller, hsmm_trainer, hsmm_artifacts


def build_examples_by_dataset(
    selected_samples: Dict[str, List],
    device: torch.device,
    trace_extractor: TraceExtractor,
    difficulty_controller: DifficultyController,
    hsmm_trainer: HSMMTrainer,
    hsmm_artifacts,
) -> Dict[str, List[Dict]]:
    amr_parser = AMRProxyParser()
    kb_backend = CompositeKBBackend()
    path_encoder = GraphPathEncoder().to(device)
    examples_by_dataset: Dict[str, List[Dict]] = {}
    for dataset_name, samples in selected_samples.items():
        examples_by_dataset[dataset_name] = [
            build_conditioned_example(
                sample,
                parser=amr_parser,
                kb_backend=kb_backend,
                path_encoder=path_encoder,
                difficulty_controller=difficulty_controller,
                hsmm_trainer=hsmm_trainer,
                hsmm_artifacts=hsmm_artifacts,
                trace_extractor=trace_extractor,
            )
            for sample in samples
        ]
    return examples_by_dataset


def build_supervised_pool(
    dataset_name: str,
    examples_by_dataset: Dict[str, List[Dict]],
) -> List[Dict]:
    return list(examples_by_dataset.get(dataset_name, []))


def train_model_for_dataset(
    dataset_name: str,
    examples_by_dataset: Dict[str, List[Dict]],
    hsmm_trainer: HSMMTrainer,
    hsmm_artifacts,
    device: torch.device,
    pretrain_epochs: int,
) -> tuple[ControllableGenerator, RewardSuite]:
    supervised_examples = build_supervised_pool(dataset_name, examples_by_dataset)
    if not supervised_examples:
        raise ValueError(f"No supervised examples available for dataset '{dataset_name}'")

    model = ControllableGenerator().to(device)
    pretrain(
        model,
        supervised_examples,
        device=device,
        epochs=max(1, pretrain_epochs),
        batch_size=2,
        lr=1e-5,
    )

    reward_suite = RewardSuite(hsmm_trainer=hsmm_trainer, hsmm_artifacts=hsmm_artifacts)
    return model, reward_suite


def main() -> None:
    parser = argparse.ArgumentParser(description="Export four-dataset DCQG predictions to a single workbook")
    parser.add_argument("--data_dir", default="data")
    parser.add_argument("--output_dir", default="output")
    parser.add_argument("--samples_per_dataset", type=int, default=30)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--pretrain_epochs", type=int, default=12)
    parser.add_argument("--num_candidates", type=int, default=6)
    parser.add_argument("--state_path", default=None)
    parser.add_argument("--overwrite", action="store_true")
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)
    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    state_path = args.state_path or os.path.join(args.output_dir, "selection_state.json")
    append_mode = not args.overwrite

    existing_sheets = read_xlsx(os.path.join(args.output_dir, "four_dataset_predictions.xlsx")) if append_mode else {}
    state: Dict[str, int] = {}
    if os.path.exists(state_path):
        with open(state_path, "r", encoding="utf-8") as handle:
            state = json.load(handle)

    selected_samples: Dict[str, List] = {}
    offsets: Dict[str, int] = {}
    for dataset_name, loader in DATASET_LOADERS.items():
        existing_count = len(existing_sheets.get(dataset_name, [])) if append_mode else 0
        start_idx = int(state.get(dataset_name, existing_count))
        offsets[dataset_name] = start_idx
        split = loader(args.data_dir, max_train=start_idx + args.samples_per_dataset, max_dev=0)
        selected_samples[dataset_name] = split["train"][start_idx : start_idx + args.samples_per_dataset]

    trace_extractor, difficulty_controller, hsmm_trainer, hsmm_artifacts = fit_shared_structural_models(
        args.data_dir,
        selected_samples,
        device,
    )
    examples_by_dataset = build_examples_by_dataset(
        selected_samples,
        device,
        trace_extractor,
        difficulty_controller,
        hsmm_trainer,
        hsmm_artifacts,
    )

    sheets: Dict[str, List[Dict[str, object]]] = dict(existing_sheets)
    export_summary: Dict[str, Dict[str, object]] = {}
    for dataset_name, examples in examples_by_dataset.items():
        if not examples:
            continue
        model, reward_suite = train_model_for_dataset(
            dataset_name,
            examples_by_dataset,
            hsmm_trainer,
            hsmm_artifacts,
            device,
            pretrain_epochs=args.pretrain_epochs,
        )
        records = generate_records(
            model,
            examples,
            device=device,
            limit=len(examples),
            reward_suite=reward_suite,
            num_candidates=max(2, args.num_candidates),
        )
        for record in records:
            record["sample_id"] = record.get("sample_id") or ""
        for record, example in zip(records, examples):
            record["sample_id"] = example["sample"].sample_id
        new_rows = build_sheet_rows(records)
        sheets.setdefault(dataset_name, [])
        sheets[dataset_name].extend(new_rows)
        export_summary[dataset_name] = {
            "count": len(new_rows),
            "output_sheet": dataset_name,
            "pretrain_epochs": args.pretrain_epochs,
            "num_candidates": max(2, args.num_candidates),
            "start_index": offsets[dataset_name],
            "end_index": offsets[dataset_name] + len(new_rows),
        }
        offsets[dataset_name] = offsets[dataset_name] + len(new_rows)

    workbook_path = os.path.join(args.output_dir, "four_dataset_predictions.xlsx")
    headers = [
        "sample_id",
        "context",
        "answer",
        "reference_question",
        "generated_question",
        "target_difficulty",
        "predicted_hop",
        "bleu_4",
        "rouge_l",
        "meteor",
        "bertscore",
    ]
    write_csv_outputs(args.output_dir, sheets, headers)
    write_xlsx_pandas(workbook_path, sheets, headers)

    with open(state_path, "w", encoding="utf-8") as handle:
        json.dump(offsets, handle, indent=2)

    summary_path = os.path.join(args.output_dir, "four_dataset_predictions.summary.json")
    with open(summary_path, "w", encoding="utf-8") as handle:
        json.dump({"workbook_path": workbook_path, "datasets": export_summary, "state_path": state_path}, handle, indent=2)

    print(json.dumps({"workbook_path": workbook_path, "datasets": export_summary, "state_path": state_path}, indent=2))


if __name__ == "__main__":
    main()


Writing /content/export_four_dataset_workbook.py


In [17]:
!python -m py_compile /content/main.py /content/graphs_build.py /content/skeletons.py /content/hsmm.py /content/generator.py /content/evaluation.py /content/export_four_dataset_workbook.py


In [18]:
import gc, torch

# delete large variables if you have names
# del model, tokenizer, outputs, ...

gc.collect()
torch.cuda.empty_cache()


In [22]:
%cd /content
!python export_four_dataset_workbook.py   --data_dir {DATA_DIR}   --output_dir {OUTPUT_DIR}   --samples_per_dataset 30   --num_candidates 1   --state_path $OUTPUT_DIR/selection_state.json


/content
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 139kB/s]
vocab.json: 899kB [00:00, 36.6MB/s]
merges.txt: 456kB [00:00, 90.0MB/s]
tokenizer.json: 1.36MB [00:00, 113MB/s]
config.json: 1.63kB [00:00, 1.07MB/s]
model.safetensors: 100% 1.02G/1.02G [00:07<00:00, 141MB/s]
Loading weights: 100% 513/513 [00:00<00:00, 843.21it/s, Materializing param=model.shared.weight]
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The following generation flags are not valid and may be ignored: ['t

In [ ]:
%cd /content


/content/flowchart_dcqg
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 129kB/s]
vocab.json: 899kB [00:00, 33.0MB/s]
merges.txt: 456kB [00:00, 106MB/s]
tokenizer.json: 1.36MB [00:00, 116MB/s]
config.json: 1.63kB [00:00, 5.55MB/s]
model.safetensors: 100% 1.02G/1.02G [00:09<00:00, 103MB/s]
Loading weights: 100% 513/513 [00:00<00:00, 885.53it/s, Materializing param=model.shared.weight]
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
config.json: 100% 482/482 [00:00<00:00, 2.64MB/s]
toke

In [ ]:
import json
summary_path = '/content/output/four_dataset_predictions.summary.json'
with open(summary_path, 'r', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2))


In [ ]:
import os
workbook_path = '/content/output/four_dataset_predictions.xlsx'
print(workbook_path)
print('exists:', os.path.exists(workbook_path))
print('size_bytes:', os.path.getsize(workbook_path) if os.path.exists(workbook_path) else 0)
